# Tool 0: one participant, the whole pipeline

Explore **one recording** through every step of the Dream-Toolkit pipeline, with all the results shown here.
Use it to **diagnose a participant** (why was this night excluded? what does this artefact look like?) and to
**learn the pipeline** before running the batch tools on a whole database.

**How to use it**
1. Pick the **study folder** (the one holding `config_param/`, `derivatives/` and the `reports_*` folders; the
   recordings are found in any sub-folder), then the **participant**.
2. Go through the steps in order. Each tab says what the step does, which batch tool runs it on the whole
   database, and what to look at. Press its **Run** button.
3. The **epoch explorer** below the steps shows the night epoch by epoch as soon as the recording is loaded
   (step 1). You can rescore an epoch or add an annotation there.

**What it writes** (nothing else is ever modified):
- `<hypnogram>_rescored.txt` + `<hypnogram>_rescore_log.tsv` beside the hypnogram, when you save a rescoring;
- `<recording>_manual_events.tsv` beside the recording, when you add an annotation (the same file as tool 8's;
  tool 7 can include it in its event flagging);
- an inspection log and report in `reports_live_explore/` (untick *Keep an inspection report* to write nothing
  there).

Step 8 is being built (tool-0 redesign): its tab says so.

In [ ]:
try:
    import os
    import sys
    import io
    import json
    import base64
    from html import escape
    import warnings
    from datetime import datetime
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.patheffects as mpe
    import matplotlib.colors as mcolors
    from matplotlib.lines import Line2D
    import mne
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output
    from ipyfilechooser import FileChooser
    from scipy.signal import spectrogram as scipy_spectrogram

    # Shared libraries (tools/): found whether Voila is launched from the repo root or from tools/.
    _here = os.getcwd()
    for _cand in [_here, os.path.join(_here, 'tools'), os.path.join(os.path.dirname(_here), 'tools'),
                  os.path.join(_here, '..', 'tools')]:
        if os.path.isfile(os.path.join(_cand, 'edf_io_lib.py')):
            _cand = os.path.abspath(_cand)
            if _cand not in sys.path:
                sys.path.insert(0, _cand)
            break
    import edf_io_lib as EIO                    # EDF reading, the same code as the batch tools
    import macrostructure_lib as ML             # tool 5's metric registry and computations
    import quality_lib as QL                    # tool 6's signal-quality metrics and figures
    import preprocessing_lib as PL              # tool 7's preprocessing, flagging, tables and figures
    import autoreject_lib as AR                 # tool 8bis's decision rule, tables and figures
    import qc_rejected_epochs_lib as L          # tool 8's display (montage, detail panel, annotations)
    import participant_selection_lib as PSL     # exclusion registry
except ImportError as e:
    print('⚠️ Import error, a package is missing:', e)
else:
    print('✅ Packages imported successfully!')

# ipyevents gives keyboard navigation in the explorer; optional (the buttons do everything without it).
try:
    from ipyevents import Event as _IpyEvent
    _HAS_IPYEVENTS = True
except Exception:
    _HAS_IPYEVENTS = False

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

In [ ]:
# ----- Steps, their status, and what a change makes out of date -----
SECTIONS = [('1', 'Load'), ('2', 'Header'), ('3', 'Scoring & events'), ('4', 'Macrostructure'),
            ('5', 'Quality'), ('6', 'Preprocessing'), ('7', 'Decision'), ('8', 'Spectral')]
# A rescoring or an annotation changes what these steps are computed from (agreement with GSSC, stage
# durations, per-stage amplitude thresholds, event flags): they are marked OUT OF DATE, never recomputed
# behind the user's back.
AFTER_RESCORING = ['3', '4', '5', '6', '7', '8']
AFTER_ANNOTATION = ['3', '6', '7', '8']
DOWNSTREAM = {'1': ['3', '4', '5', '6', '7', '8'], '6': ['7', '8'], '7': ['8']}

STATE = {}                                  # the loaded participant (cleared on every participant change)
STATUS = {k: 'not_run' for k, _ in SECTIONS}
STATUS_WHY = {k: '' for k, _ in SECTIONS}
REPORT_BLOCKS = {}                          # step key -> HTML fragments shown in the inspection report
BADGES = {}                                 # step key -> its status widget (filled when the tabs are built)
TABS = None                                 # the Tab widget (its titles carry the status mark)


def badge_html(key):
    s = STATUS[key]
    if s == 'up_to_date':
        return ('<span style="background:#e8f5e9;color:#1b5e20;padding:2px 8px;border-radius:3px;">'
                'up to date</span>')
    if s == 'out_of_date':
        why = f': {STATUS_WHY[key]}' if STATUS_WHY[key] else ''
        return ('<span style="background:#fff3e0;color:#e65100;padding:2px 8px;border-radius:3px;">'
                f'<b>OUT OF DATE</b>{why}. Run it again.</span>')
    return '<span style="background:#f5f5f5;color:#757575;padding:2px 8px;border-radius:3px;">not run</span>'


def refresh_badges():
    for k, w in BADGES.items():
        w.value = badge_html(k)
    if TABS is not None:
        mark = {'up_to_date': ' ✓', 'out_of_date': ' ⚠', 'not_run': ''}
        for i, (k, name) in enumerate(SECTIONS):
            TABS.set_title(i, f'{k} {name}{mark[STATUS[k]]}')


def mark_out_of_date(keys, why):
    # Only a step that HAS results can become out of date; a step never run stays 'not run'.
    for k in keys:
        if STATUS[k] == 'up_to_date':
            STATUS[k] = 'out_of_date'
            STATUS_WHY[k] = why
    refresh_badges()


def mark_up_to_date(key):
    STATUS[key] = 'up_to_date'
    STATUS_WHY[key] = ''
    mark_out_of_date(DOWNSTREAM.get(key, []), f'step {key} was run again')


def unsaved_rescoring():
    """Number of epochs rescored since the last save (0 when nothing is loaded)."""
    if 'hypno_work' not in STATE:
        return 0
    return int((STATE['hypno_work'] != STATE['hypno_saved']).sum())


def guard_unsaved(what):
    """True when `what` may proceed. Unsaved rescoring blocks the FIRST attempt (warning); the same action
    repeated discards it, like tool 8's unsaved-changes guard."""
    n = unsaved_rescoring()
    if n == 0 or STATE.get('discard_armed') == what:
        return True
    STATE['discard_armed'] = what
    html_guard.value = (f'<p style="background:#fff3e0;color:#e65100;padding:4px 8px;"><b>⚠ {n} rescored '
                        f'epoch(s) not saved.</b> Save them in the explorer, or {what} again to discard them.</p>')
    return False


def reset_state():
    STATE.clear()
    for k in STATUS:
        STATUS[k] = 'not_run'
        STATUS_WHY[k] = ''
    REPORT_BLOCKS.clear()
    refresh_badges()


# ----- Figures -----
def fig_to_png(fig, tight=True):
    """PNG bytes of a figure (closed afterwards). tight=False keeps the figure's own size and margins: the
    explorer's navigator and montage need it so that their plotting areas stay exactly one above the other."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight' if tight else None)
    plt.close(fig)
    return buf.getvalue()


def png_html(png, alt=''):
    return (f'<img alt="{alt}" style="max-width:100%;" '
            f'src="data:image/png;base64,{base64.b64encode(png).decode()}">')


def training_box(what, batch, look):
    """The 'About this step' box at the top of each tab (training aid for a new user)."""
    return widgets.HTML(
        '<details open style="background:#f4f8fb;border-left:4px solid #2a6099;padding:6px 10px;'
        'margin:4px 0 8px 0;"><summary><b>About this step</b></summary>'
        f'<p style="margin:4px 0;"><b>What it does.</b> {what}</p>'
        f'<p style="margin:4px 0;"><b>On the whole database.</b> {batch}</p>'
        f'<p style="margin:4px 0;"><b>What to look at.</b> {look}</p></details>')


def df_html(df):
    return df.to_html(index=False, border=0, na_rep='', escape=True,
                      classes='dtk').replace('<table', '<table style="border-collapse:collapse;font-size:0.9em;"')


# ----- Inspection log + report (reports_live_explore/) -----
LOG_COLUMNS = ['file_id', 'time', 'step', 'action', 'details']


def report_folder():
    """<study>/reports_live_explore/<sub-folder of the recording>/ (mirrors the recordings' tree)."""
    study, edf = Path(STATE['study']), Path(STATE['edf'])
    try:
        sub = edf.parent.relative_to(study)
    except ValueError:
        sub = Path('.')
    return study / 'reports_live_explore' / sub


def log_action(step, action, details=''):
    """One action of the inspection: appended to {file_id}_inspection_log.tsv (kept across sessions, data
    first), then the report is rewritten. Non-fatal: a failure is shown, never raised."""
    if not cb_report.value or 'edf' not in STATE:
        return
    try:
        folder = report_folder()
        folder.mkdir(parents=True, exist_ok=True)
        path = folder / f"{STATE['fid']}_inspection_log.tsv"
        row = pd.DataFrame([{'file_id': STATE['fid'], 'time': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                             'step': step, 'action': action, 'details': details}], columns=LOG_COLUMNS)
        row.to_csv(path, sep='\t', index=False, mode='a', header=not path.exists())
    except Exception as e:
        html_report_info.value = f'<span style="color:#c62828;">⚠ inspection log not written: {e}</span>'
        return
    write_report()


def write_report():
    """Rewrite {file_id}_inspection_report.html: participant, step statuses, the whole log, and each
    step's latest results. Non-fatal."""
    try:
        folder = report_folder()
        log_path = folder / f"{STATE['fid']}_inspection_log.tsv"
        log = pd.read_csv(log_path, sep='\t') if log_path.exists() else pd.DataFrame(columns=LOG_COLUMNS)
        status_rows = ''.join(f'<tr><td style="padding:2px 10px;">{k} {n}</td><td>{badge_html(k)}</td></tr>'
                              for k, n in SECTIONS)
        blocks = ''
        for k, n in SECTIONS + [('explorer', 'Epoch explorer')]:
            if REPORT_BLOCKS.get(k):
                title = n if k == 'explorer' else f'{k}. {n}'
                blocks += f'<h2>{title}</h2>' + ''.join(REPORT_BLOCKS[k])
        html = ('<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8">'
                f"<title>Inspection {STATE['fid']}</title>"
                '<style>body{font-family:Arial,sans-serif;max-width:1300px;margin:auto;padding:10px;}'
                'td,th{padding:2px 8px;border-bottom:1px solid #eee;text-align:left;}</style></head><body>'
                f"<h1>Inspection of {STATE['fid']}</h1>"
                f"<p>Recording: <code>{STATE['edf']}</code><br>Study folder: <code>{STATE['study']}</code><br>"
                f"Report written {datetime.now().strftime('%Y-%m-%d %H:%M')} by tool 0.</p>"
                f"{STATE.get('registry_html', '')}"
                f'<h2>Steps</h2><table>{status_rows}</table>'
                f'<h2>What was done (all sessions)</h2>{df_html(log)}{blocks}</body></html>')
        out = folder / f"{STATE['fid']}_inspection_report.html"
        out.write_text(html, encoding='utf-8')
        html_report_info.value = f'<small>Inspection report: {out}</small>'
    except Exception as e:
        html_report_info.value = f'<span style="color:#c62828;">⚠ inspection report not written: {e}</span>'


# The toolkit's display conventions (quality_lib: REM red, spectrogram palette; macrostructure_lib: the colour
# family of the scored events), shared with the batch tools' figures.
REM_COLOR = QL.REM_COLOR
event_family = ML.event_family

## Study folder and participant

In [ ]:
fc_data = FileChooser(os.getcwd())
fc_data.show_only_dirs = True
fc_data.title = ('<b>Study folder</b>: the folder holding <code>config_param/</code>, <code>derivatives/</code> '
                 'and the <code>reports_*</code> folders (the recordings are found in any sub-folder)')
dd_part = widgets.Dropdown(description='Participant:', options=[], value=None,
                           style={'description_width': '90px'}, layout=widgets.Layout(width='420px'))
cb_report = widgets.Checkbox(value=True, description='Keep an inspection report (reports_live_explore/)',
                             indent=False, layout=widgets.Layout(width='420px'))
html_study = widgets.HTML('')
html_part = widgets.HTML('')
html_report_info = widgets.HTML('')
html_guard = widgets.HTML('')          # unsaved-rescoring warning

STUDY = {}          # study-wide scan: recordings, configuration files, companion suffixes (tool 7 rules)
NONE = '(none)'


def _in_output_tree(path, study):
    # derivatives/ and reports_* hold outputs, never recordings
    try:
        first = Path(path).relative_to(study).parts[0]
    except Exception:
        return False
    return first == 'derivatives' or first.startswith('reports_')


def scan_study(study):
    """Recordings + configuration files of the study + the companion suffixes tool 7 would pick."""
    study = Path(study)
    edfs, dup = {}, []
    for f in EIO.list_edf_files(study):
        if _in_output_tree(f, study):
            continue
        if f.stem in edfs:
            dup.append(f.stem)
            continue
        edfs[f.stem] = f
    files = [f for f in study.rglob('*') if f.suffix.lower() in ('.txt', '.csv')
             and not _in_output_tree(f, study)]
    txt = [f for f in files if f.suffix.lower() == '.txt']
    csv = [f for f in files if f.suffix.lower() == '.csv']
    edf_list = list(edfs.values())
    hyp_counts = EIO.companion_suffix_counts(edf_list, txt)
    hyp_suffix, _ = EIO.select_hypno_suffix(hyp_counts)
    txt_suffix, _ = EIO.select_event_suffix(
        EIO.companion_suffix_counts(edf_list, [f for f in txt if 'event' in f.name.lower()]))
    csv_suffix, _ = EIO.select_event_suffix(EIO.companion_suffix_counts(edf_list, csv))
    # tool 5's lights-off / lights-on export (*_Summary_Export.txt): the most frequent 'summary' suffix
    summary_suffix, _ = EIO.select_event_suffix(
        EIO.companion_suffix_counts(edf_list, [f for f in txt if 'summary' in f.name.lower()]))
    # participant table at the study root (tool 5 reads lights-off / lights-on columns from it)
    subj_info, subj_col = None, None
    for name in ('participants.tsv', 'participants.csv', 'participants.xlsx'):
        if (study / name).exists():
            subj_info, _err = ML.load_subject_info(str(study / name))
            if subj_info is not None:
                guess = [c for c in subj_info.columns if ML.norm(c) in ('file_id', 'fileid', 'sub_id', 'subid', 'id',
                                                                        'participant', 'participant_id', 'subject')]
                subj_col = guess[0] if guess else list(subj_info.columns)[0]
            break
    cfg_dir = study / 'config_param'
    config, config_msg = None, "no <code>config_param/remap_reref_persubject.json</code> (tool 2): channels auto-detected"
    if (cfg_dir / 'remap_reref_persubject.json').exists():
        try:
            config = EIO.load_json_lenient(cfg_dir / 'remap_reref_persubject.json')
            config_msg = f"channel configuration of tool 2 read ({len(config)} participant(s))"
        except Exception as e:
            config_msg = f'<span style="color:#c62828;">⚠ tool 2 configuration unreadable: {e}</span>'
    event_remap, event_msg = {}, 'no <code>config_param/event_remap.json</code> (tool 4): raw event labels shown'
    if (cfg_dir / 'event_remap.json').exists():
        try:
            event_remap = EIO.load_event_remap(cfg_dir / 'event_remap.json')
            event_msg = f'event labels of tool 4 read ({len(event_remap)} label(s))'
        except Exception as e:
            event_msg = f'<span style="color:#c62828;">⚠ tool 4 event remap unreadable: {e}</span>'
    STUDY.update(study=study, edfs=edfs, dup=dup, hypno_suffix=hyp_suffix, summary_suffix=summary_suffix,
                 subj_info=subj_info, subj_col=subj_col,
                 txt_suffix=txt_suffix or '_ScoredEvents_Export.txt', csv_suffix=csv_suffix or '_event_xml.csv',
                 config=config, config_msg=config_msg, event_remap=event_remap, event_msg=event_msg,
                 custom_stages=L.load_custom_stages(study))


def _on_study(chooser=None):
    try:
        STUDY.clear()
        html_study.value = ''
        dd_part.unobserve(_on_participant, names='value')
        dd_part.options = []
        dd_part.value = None
        dd_part.observe(_on_participant, names='value')
        reset_participant()
        if not fc_data.selected_path:
            return
        scan_study(fc_data.selected_path)
        n = len(STUDY['edfs'])
        dup = (f"<br>⚠ same file name in two sub-folders, only the first kept: {', '.join(STUDY['dup'])}"
               if STUDY['dup'] else '')
        html_study.value = (f"<small>{n} recording(s) found. {STUDY['config_msg']}; {STUDY['event_msg']}. "
                            f"Hypnogram suffix tool 7 would pick: <code>{STUDY['hypno_suffix']}</code>.{dup}</small>")
        dd_part.unobserve(_on_participant, names='value')
        dd_part.options = sorted(STUDY['edfs'])
        dd_part.value = None
        dd_part.observe(_on_participant, names='value')
    except Exception as e:
        html_study.value = f'<span style="color:#c62828;">Study folder error: {e}</span>'


def _on_participant(change):
    try:
        if not guard_unsaved('change the participant'):
            dd_part.unobserve(_on_participant, names='value')
            dd_part.value = change['old']          # stay on the participant whose rescoring is unsaved
            dd_part.observe(_on_participant, names='value')
            return
        reset_participant()
        if change['new'] is None:
            return
        prefill_participant(change['new'])
    except Exception as e:
        html_part.value = f'<span style="color:#c62828;">Could not read this recording: {e}</span>'


fc_data.register_callback(_on_study)
dd_part.observe(_on_participant, names='value')

## Steps

In [ ]:
# ===== Step 1: Load & configuration =====
btn_run1 = widgets.Button(description='Load the recording', icon='play', button_style='primary',
                          layout=widgets.Layout(width='220px', height='36px'))
html_cfg1 = widgets.HTML('')          # what was found for this participant (header only, no signal)
box_eeg1 = widgets.VBox([])           # EEG checkboxes, only when tool 2's configuration has no entry
eeg_cbs = {}
CTX_SLOTS = [('eog_left', 'EOG left'), ('eog_right', 'EOG right'), ('emg', 'EMG'), ('ecg', 'ECG')]
dd_ctx = {role: widgets.Dropdown(description=label, options=[NONE], value=NONE,
                                 style={'description_width': '70px'}, layout=widgets.Layout(width='250px'))
          for role, label in CTX_SLOTS}
html_ctx1 = widgets.HTML('')
dd_hypno = widgets.Dropdown(description='Hypnogram:', options=[NONE], value=NONE,
                            style={'description_width': '90px'}, layout=widgets.Layout(width='480px'))
html_hypno1 = widgets.HTML('')
# a hypnogram the detection missed (another name, another folder): chosen by hand
OTHER_FILE = 'Other file…'
fc_hypno = FileChooser(os.getcwd())
fc_hypno.title = '<b>Hypnogram file</b> (one stage label per 30 s epoch, like the detected ones)'
box_hypno_other = widgets.VBox([
    fc_hypno,
    widgets.HTML('<small style="color:#e65100;">⚠ The batch tools will not find this file: they read '
                 '<code>&lt;recording&gt;&lt;suffix&gt;.txt</code> beside the recording. Rename or copy it there '
                 'for them.</small>')],
    layout=widgets.Layout(display='' if dd_hypno.value == OTHER_FILE else 'none'))
dd_hypno.observe(lambda c: setattr(box_hypno_other.layout, 'display', '' if c['new'] == OTHER_FILE else 'none'),
                 names='value')
txt_custom = widgets.Text(description='Custom stages:',
                          placeholder='non-AASM labels kept, e.g. N4 (from config_param/custom_stages.json)',
                          style={'description_width': '110px'}, layout=widgets.Layout(width='480px'))
out_sec1 = widgets.Output()


def registry_status_html(study, fid):
    reg, warn = PSL.load_registry(study)
    excluded, reason = PSL.status_of(PSL.effective_status(reg), fid)
    if warn:
        return f'<p style="color:#e65100;">⚠ {warn}</p>'
    if excluded:
        return (f'<p style="background:#ffebee;color:#b71c1c;padding:4px 8px;"><b>EXCLUDED</b> in the '
                f'exclusion registry: {reason}</p>')
    return ('<p style="color:#1b5e20;">Included (no exclusion in <code>config_param/participant_exclusions.tsv'
            '</code>).</p>')


def prefill_participant(fid):
    """Header-only look at the chosen recording (the signal is read by 'Load the recording'): channels and
    their detected types, tool 2's configuration entry, context channels, hypnograms, event files."""
    edf = STUDY['edfs'][fid]
    STATE.update(study=STUDY['study'], edf=edf, fid=fid)
    hdr = EIO.read_edf_header_custom(str(edf), fallback_encoding='latin-1')
    chans, trans = hdr['channel'], hdr['transducer_type']
    types = EIO.detect_channel_types(chans, trans)
    cfg = EIO.participant_config(STUDY['config'], fid) if STUDY.get('config') else None
    STATE.update(header=hdr, types=types, cfg=cfg)
    STATE['registry_html'] = registry_status_html(STUDY['study'], fid)
    html_part.value = f'<small>{edf}</small>' + STATE['registry_html']

    parts = []
    # EEG channels: tool 2's remap when this participant has an entry, else detection (editable)
    eeg_cbs.clear()
    if cfg is not None and cfg.get('remap'):
        remap = cfg['remap']
        rows = ''.join(f'<tr><td>{o}</td><td>&rarr; {n}</td><td>{"" if o in chans else "⚠ not in the file"}</td></tr>'
                       for o, n in remap.items())
        parts.append(f"<b>EEG channels</b> from tool 2's configuration (re-reference for step 6: "
                     f"<code>{cfg.get('ref_channels')}</code>):<table>{rows}</table>")
        box_eeg1.children = ()
    else:
        for ch in chans:
            eeg_cbs[ch] = widgets.Checkbox(value=(types[ch] == 'EEG'), description=ch, indent=False,
                                           layout=widgets.Layout(width='140px'))
        box_eeg1.children = (
            widgets.HTML("<b>EEG channels</b>: this participant has no entry in tool 2's configuration, so "
                         "they are auto-detected. Tick / untick (names kept as in the file):"),
            widgets.Box(list(eeg_cbs.values()), layout=widgets.Layout(flex_flow='row wrap')))
    # context channels: declared in tool 2's configuration, else detected (editable, never saved)
    declared = (cfg or {}).get('context_channels') or {}
    if declared:
        guess, src = declared, "declared in tool 2's configuration"
    else:
        guess = EIO.detect_context_channels(pd.DataFrame({'channel': chans, 'transducer_type': trans}))
        src = ("auto-detected (no <code>context_channels</code> in tool 2's configuration): check them, "
               'tool 7 only uses declared ones')
    for role, dd in dd_ctx.items():
        dd.options = [NONE] + list(chans)
        dd.value = guess.get(role) if guess.get(role) in chans else NONE
    html_ctx1.value = f'<small>EOG / EMG / ECG context channels {src}.</small>'
    # hypnograms: every .txt named after the recording; tool 7's choice pre-selected
    cands = sorted(f.name[len(edf.stem):] for f in edf.parent.iterdir()
                   if f.suffix.lower() == '.txt' and os.path.normcase(f.name).startswith(os.path.normcase(edf.stem)))
    hyp = [s for s in cands if 'event' not in s.lower()] + [s for s in cands if 'event' in s.lower()]
    dd_hypno.options = [NONE] + hyp + [OTHER_FILE]
    best = STUDY.get('hypno_suffix')
    dd_hypno.value = best if best in hyp else (hyp[0] if hyp and 'event' not in hyp[0].lower() else NONE)
    fc_hypno.reset(path=str(edf.parent))
    pick = f'or "{OTHER_FILE}" to choose a file the detection missed.'
    html_hypno1.value = (f"<small>Suffix tool 7 would use for the whole study: <code>{best}</code>. "
                         "Pick another one to look at a different scoring (e.g. tool 3bis's automatic one), "
                         f'{pick}</small>' if best else
                         f'<small style="color:#e65100;">No hypnogram found beside the recordings: pick {pick[3:]}</small>')
    # events: which companion the TXT -> CSV -> XML chain will read
    txt, csv, xml = EIO.event_companion_paths(edf, STUDY['txt_suffix'], STUDY['csv_suffix'])
    found = [f'{k} <code>{Path(p).name}</code>' for k, p in (('TXT', txt), ('CSV', csv), ('XML', xml)) if p]
    parts.append('<b>Scored events</b>: ' + (', '.join(found) + ' (read in this order of priority, like tools 5 and 7)'
                                             if found else 'no event file beside the recording'))
    sf_eeg = sorted({sf for ch, sf in zip(chans, hdr['sampling_frequency']) if types[ch] == 'EEG'})
    parts.insert(0, f"<b>{edf.name}</b>: {len(chans)} channels in the file, EEG sampled at {', '.join(sf_eeg) or '?'} Hz.")
    html_cfg1.value = '<br>'.join(parts)
    txt_custom.value = ', '.join(STUDY.get('custom_stages', []))
    with out_sec1:
        clear_output()
        print('Check the configuration above, then click "Load the recording".')


def load_recording():
    """Read the signal like tool 7 (EEG as recorded: no re-reference, no filter), the context channels,
    the hypnogram (tool 7's length rule), the scored events and the manual annotations."""
    edf, fid, cfg = Path(STATE['edf']), STATE['fid'], STATE['cfg']
    custom = L.parse_custom_field(txt_custom.value)
    notes = []
    if cfg is not None and cfg.get('remap'):
        raw = EIO.read_eeg_raw(edf, cfg['remap'])
    else:
        sel = [ch for ch, cb in eeg_cbs.items() if cb.value]
        if not sel:
            raise ValueError('no EEG channel selected')
        raw = mne.io.read_raw_edf(str(edf), preload=False, encoding='latin-1', include=sel, verbose=False)
        raw, _ = EIO.drop_suffix_duplicates(raw)
        raw.load_data()
    if not len(raw.ch_names):
        raise ValueError('none of the EEG channels is in the file')
    sf = float(raw.info['sfreq'])
    n_sig = int(np.floor(raw.n_times / sf / 30))
    # hypnogram, with tool 7's length rule
    hpath, status = None, 'none'
    by_hand = dd_hypno.value == OTHER_FILE
    if by_hand:
        if not fc_hypno.selected:
            raise ValueError(f'"{OTHER_FILE}" is chosen but no hypnogram file is selected')
        hpath = Path(fc_hypno.selected)
    elif dd_hypno.value != NONE:
        hpath = edf.with_name(f'{edf.stem}{dd_hypno.value}')
    if hpath is not None:
        labels = EIO.load_hypno_labels(hpath)
        n_hyp = len(labels)
        status = EIO.hypno_length_status(n_hyp, n_sig)
        if status == 'trim':
            labels = labels[:n_sig]
            notes.append(f'The recording ends inside its last 30 s epoch: the last score is dropped '
                         f'({n_hyp} &rarr; {n_sig} epochs), as tool 7 does.')
        elif status == 'mismatch':
            notes.append(f'<span style="color:#c62828;"><b>⚠ Length mismatch</b>: the hypnogram has {n_hyp} epochs, '
                         f'the signal {n_sig} complete 30 s epochs. <b>Tool 7 skips this participant.</b> '
                         'Here the hypnogram is cut / padded with "?" so the night can still be looked at.</span>')
            labels = (labels[:n_sig] if n_hyp > n_sig else
                      np.concatenate([labels, np.array(['?'] * (n_sig - n_hyp), dtype='<U10')]))
    else:
        labels = np.array(['?'] * n_sig, dtype='<U10')
        notes.append('No hypnogram: every epoch is "?" (you can score the night in the explorer).')
    unknown = sorted(set(labels) - set(L.AASM_STAGES) - {'MT', '?'} - set(custom))
    if unknown:
        n_unk = int(np.isin(labels, unknown).sum())
        notes.append(f'<span style="color:#e65100;">⚠ {n_unk} epoch(s) carry a label outside W/N1/N2/N3/R: '
                     f'{", ".join(unknown)}. Harmonize them with tool 3, or declare them as custom stages.</span>')
    # scored events (TXT -> CSV -> XML), labels translated with tool 4's event_remap.json
    ev, ev_src = EIO.load_events(edf, STUDY['txt_suffix'], STUDY['csv_suffix'])
    onsets = ML.events_for_display(ev, STUDY.get('event_remap'))     # the same table as tool 5's report
    manual = L.load_manual_events(edf.parent, fid)
    # context channels (EOG / EMG / ECG), read and display-filtered like tool 7's companion
    ctx_map = {role: dd.value for role, dd in dd_ctx.items() if dd.value != NONE}
    ctx = None
    if ctx_map:
        ctx_raw = EIO.read_context_raw(edf, ctx_map, sf)
        if ctx_raw is not None:
            cep = mne.make_fixed_length_epochs(ctx_raw, duration=30, preload=True, verbose=False)
            ctx = {'data_uV': cep.get_data() * 1e6, 'sfreq': float(cep.info['sfreq']), 'labels': list(cep.ch_names)}
    STATE.update(raw=raw, sf=sf, n_sig=n_sig, hypno_path=hpath, hypno_status=status, hypno_by_hand=by_hand,
                 hypno_orig=labels.copy(), hypno_work=labels.copy(), hypno_saved=labels.copy(),
                 unknown_labels=unknown, discard_armed=None,
                 events=ev, events_source=ev_src, onsets=onsets, manual_events=manual,
                 ctx=ctx, ctx_map=ctx_map, notes=notes, custom_stages=custom)
    build_display('raw')


def build_display(mode):
    """Epochs of the displayed signal ('raw' = as recorded, 'bp' = display band-pass 0.3-35 Hz, 'preproc' =
    step 6's epochs: tool 7's preprocessing with its flags, the deselected channels left out as tool 8 does),
    cut like tool 7 (30 s from t = 0, as many as the hypnogram), their PSD (detail panel) and the navigator
    spectrogram. A band-pass is for the eyes only: it never reaches a computation of steps 2-8."""
    if mode == 'preproc':
        r = STATE['preproc']
        P = L.participant_from_epochs(r['epochs'].copy())
        s = r['settings']
        smoothing = {'enabled': s['ss'], 'method': 'median+lowess' if s['ss_median'] else 'lowess',
                     'median_span_hz': s['ss_median_span'] if s['ss_median'] else None, 'lowess_span_hz': s['ss_span']}
        freqs, psds = L.compute_psds(P['epochs'], fmax=s['fit_fmax'], smoothing=smoothing)   # as tool 8 with the sidecar
        STATE.update(P=P, freqs=freqs, psds=psds, display_mode=mode, n=r['n_epochs'])
        compute_nav_spectrogram()
        return
    raw = STATE['raw']
    if mode == 'bp':
        raw = raw.copy().filter(0.3, min(35.0, STATE['sf'] / 2 - 0.5), verbose=False)
    epochs = mne.make_fixed_length_epochs(raw, duration=30, preload=False, verbose=False)
    epochs.drop_bad(verbose=False)            # epochs over a BAD annotation are dropped, as in tool 7
    n = min(len(epochs), len(STATE['hypno_work']))
    if len(epochs) < STATE['n_sig']:
        STATE['notes'].append(f'⚠ MNE dropped {STATE["n_sig"] - len(epochs)} epoch(s) overlapping a BAD '
                              'annotation of the EDF: the epoch numbers after them are shifted.')
    epochs = epochs[:n]
    epochs.metadata = pd.DataFrame({'stage': STATE['hypno_work'][:n], 'reject_flag': np.zeros(n, dtype=bool),
                                    'reject_method': ['none'] * n})
    P = L.participant_from_epochs(epochs)
    freqs, psds = L.compute_psds(P['epochs'], fmax=45.0)
    STATE.update(P=P, freqs=freqs, psds=psds, display_mode=mode, n=n)
    compute_nav_spectrogram()


def sec1_summary():
    """Summary of what was loaded (shown in the tab and kept for the report)."""
    P, labels = STATE['P'], STATE['hypno_work']
    parts = [f"<b>Recording</b>: {STATE['n']} epochs of 30 s ({STATE['n'] * 30 / 3600:.1f} h) at "
             f"{STATE['sf']:.0f} Hz; EEG: {', '.join(P['ch_names'])}; context: "
             f"{', '.join(STATE['ctx']['labels']) if STATE['ctx'] else 'none'}."]
    hname = STATE['hypno_path'].name if STATE['hypno_path'] else 'none'
    by_hand = ''
    if STATE['hypno_by_hand']:
        by_hand = (f" (chosen by hand, in <code>{STATE['hypno_path'].parent}</code>: the batch tools will not "
                   'find it)')
    counts = pd.Series(labels).value_counts()
    order = [s for s in L.AASM_STAGES + STATE['custom_stages'] + ['MT', '?'] if s in counts.index]
    order += [s for s in counts.index if s not in order]
    stage_tab = pd.DataFrame({'stage': order, 'epochs': [int(counts[s]) for s in order],
                              'minutes': [counts[s] * 0.5 for s in order]})
    parts.append(f'<b>Hypnogram</b>: <code>{hname}</code>{by_hand}' + df_html(stage_tab))
    on = STATE['onsets']
    if on is not None and len(on):
        by = on.groupby('type').agg(n=('onset_s', 'size'), mapped=('mapped', 'first')).reset_index()
        by = by.sort_values('n', ascending=False)
        by['label'] = np.where(by['mapped'], 'canonical (tool 4)', 'raw label, not mapped')
        n_unmapped = int((~on['mapped']).sum())
        warn = (f' <span style="color:#e65100;">{n_unmapped} event(s) have a label tool 4 does not map: '
                'tool 7 ignores them.</span>' if n_unmapped else '')
        parts.append(f"<b>Scored events</b>: {len(on)} from the {STATE['events_source'].upper()} companion.{warn}"
                     + df_html(by[['type', 'n', 'label']]))
    else:
        parts.append('<b>Scored events</b>: none.')
    nm = len(STATE['manual_events'])
    parts.append(f'<b>Manual annotations</b>: {nm} (tool 8 / tool 0, <code>{STATE["fid"]}_manual_events.tsv</code>).')
    if STATE['notes']:
        parts.append('<b>Notes</b><ul>' + ''.join(f'<li>{n}</li>' for n in STATE['notes']) + '</ul>')
    return '<br>'.join(parts)


def on_run1(b):
    b.disabled = True
    try:
        with out_sec1:
            clear_output()
            if 'edf' not in STATE:
                print('Choose a study folder and a participant first.')
                return
            if not guard_unsaved('click "Load the recording"'):
                print('Not reloaded: see the warning at the top.')
                return
            print('Reading the recording…')
        STATE.pop('preproc', None)             # step 6's epochs (and step 7's decision) belong to the former load
        load_recording()
        prefill_step6()
        prefill_step7()
        html_guard.value = ''
        summary = sec1_summary()
        with out_sec1:
            clear_output()
            display(HTML(summary))
        REPORT_BLOCKS['1'] = [summary]
        mark_up_to_date('1')
        explorer_open()
        log_action('1', 'recording loaded',
                   f"hypnogram={STATE['hypno_path'].name if STATE['hypno_path'] else 'none'}"
                   f"{' (chosen by hand)' if STATE['hypno_by_hand'] else ''} ({STATE['hypno_status']}); "
                   f"events={STATE['events_source']}; context={','.join(STATE['ctx_map'].values()) or 'none'}")
    except Exception as e:
        with out_sec1:
            clear_output()
            print(f'Loading failed: {e!r}')
    finally:
        b.disabled = False


btn_run1.on_click(on_run1)
BADGES['1'] = widgets.HTML(badge_html('1'))
sec1_box = widgets.VBox([
    training_box(
        "Reads the recording and gathers its configuration: which channels are EEG (tool 2's remap), which "
        'ones are the EOG / EMG / ECG context, which hypnogram and which scored events belong to it.',
        'Tool 2 declares the channels, tool 3 harmonizes the hypnogram labels and tool 4 the event labels, for '
        'every recording of the study.',
        'Is every EEG channel found? Are the context channels on the right role? Is the hypnogram as long as '
        'the recording (tool 7 skips a participant whose hypnogram does not match)? Are the events read from '
        'the expected file, and are their labels mapped by tool 4?'),
    BADGES['1'], html_cfg1, box_eeg1,
    widgets.HTML('<b>Context channels</b> (shown under the EEG in the explorer):'),
    widgets.HBox(list(dd_ctx.values())), html_ctx1,
    dd_hypno, html_hypno1, box_hypno_other, txt_custom, btn_run1, out_sec1])

In [ ]:
# ===== Step 2: Recording header =====
btn_run2 = widgets.Button(description='Show the header', icon='play', button_style='primary',
                          layout=widgets.Layout(width='220px', height='36px'))
out_sec2 = widgets.Output()


def channel_roles():
    """Role of each channel of the file in this inspection (EEG analysed / context / not used)."""
    roles = {}
    cfg = STATE.get('cfg')
    if cfg is not None and cfg.get('remap'):
        for o, n in cfg['remap'].items():
            roles[o] = f'EEG (as {n})' if n != o else 'EEG'
    else:
        for ch, cb in eeg_cbs.items():
            if cb.value:
                roles[ch] = 'EEG'
    for role, dd in dd_ctx.items():
        if dd.value != NONE:
            roles[dd.value] = 'context ' + dict(CTX_SLOTS)[role]
    return roles


def sec2_content():
    hdr, edf = STATE['header'], Path(STATE['edf'])
    df = EIO.header_channel_table(hdr)
    roles = channel_roles()
    df.insert(1, 'role here', df['channel'].map(lambda c: roles.get(c, '')))
    df.insert(2, 'detected type', df['channel'].map(STATE['types']))
    df['dynamic_range_uV'] = (df['res_theoretical'] * pow(2, 16)).round(1)
    is_eeg = df['role here'].str.startswith('EEG')
    flags = []
    narrow = df[is_eeg & (df['dynamic_range_uV'] <= EIO.DYNAMIC_RANGE_THRESHOLD_UV)]
    for _, r in narrow.iterrows():
        flags.append(f"EEG <b>{r['channel']}</b>: dynamic range {r['dynamic_range_uV']:.0f} µV "
                     f'(≤ {EIO.DYNAMIC_RANGE_THRESHOLD_UV:.0f} µV): large slow waves and artefacts are clipped.')
    rates = sorted(set(df.loc[is_eeg, 'sampling_frequency']))
    if len(rates) > 1:
        flags.append(f"EEG channels at different rates ({', '.join(rates)} Hz): MNE reads them all at the highest.")
    filt = df.loc[is_eeg, ['highpass', 'lowpass', 'notch']].astype(str).drop_duplicates()
    if len(filt) > 1:
        flags.append('The EEG channels were not exported with the same hardware filters (see the table).')
    anon = EIO.check_anonymization(hdr['patient_id'], edf.stem)
    if anon['header_anonymized']:
        anon_txt = '<span style="color:#1b5e20;">header anonymized</span>'
    else:
        anon_txt = (f"<span style=\"color:#c62828;\"><b>{anon['anon_warning']}</b>: anonymize the recording "
                    'with tool 1bis before sharing it</span>')
        flags.append('The header holds a patient name (not shown here): run tool 1bis.')
    dur_s = hdr['n_data_records'] * hdr['duration_data_record']
    glob = pd.DataFrame({'field': ['EDF version', 'start date and time', 'duration', 'data records',
                                   'channels', 'patient field format'],
                         'value': [hdr['version'],
                                   f"{hdr['start_date']} {hdr['start_time']}",
                                   f'{dur_s / 3600:.2f} h ({dur_s:.0f} s)',
                                   f"{hdr['n_data_records']} × {hdr['duration_data_record']} s",
                                   str(hdr['n_channels']), anon['patient_id_format']]})
    # role vs detected type: a channel used as EEG (tool 2's remap) or put on a context role that the
    # detection classifies otherwise is either a detection miss or a configuration error
    expected = {'EEG': 'EEG', 'context EOG left': 'EOG', 'context EOG right': 'EOG', 'context EMG': 'EMG',
                'context ECG': 'ECG'}
    df['check'] = ''
    for i, r in df.iterrows():
        role = 'EEG' if r['role here'].startswith('EEG') else r['role here']
        if role in expected and r['detected type'] != expected[role]:
            df.at[i, 'check'] = f"⚠ used as {role.replace('context ', '')}"
            flags.append(f"<b>{r['channel']}</b> is used as {role} here but detected as {r['detected type']}: "
                         "check tool 2's configuration (or the channel's name / transducer type).")
    cols = ['channel', 'role here', 'transducer_type', 'dimension', 'sampling_frequency', 'physical_min',
            'physical_max', 'dynamic_range_uV', 'res_theoretical', 'highpass', 'lowpass', 'notch']
    if (df['check'] != '').any():
        cols.insert(2, 'check')
    df[['highpass', 'lowpass', 'notch']] = df[['highpass', 'lowpass', 'notch']].astype(object).where(
        df[['highpass', 'lowpass', 'notch']].notna(), '')
    flag_html = ('<ul>' + ''.join(f'<li>{f}</li>' for f in flags) + '</ul>' if flags
                 else '<p style="color:#1b5e20;">Nothing to report.</p>')
    # one table per detected type (edf_io_lib.detect_channel_types: transducer type OR name), so a detection
    # miss shows at a glance; an empty EEG / EOG / EMG / ECG group is said, an empty "other" is left out
    tables = ''
    for typ in ('EEG', 'EOG', 'EMG', 'ECG', 'other'):
        sub = df[df['detected type'] == typ]
        if not len(sub):
            if typ != 'other':
                tables += f'<p style="margin:8px 0 2px 0;"><b>{typ}</b>: none detected</p>'
            continue
        tables += (f'<p style="margin:8px 0 2px 0;"><b>{typ}</b>: {len(sub)} channel{"s" if len(sub) > 1 else ""}</p>'
                   + df_html(sub[cols]))
    return (f'<p>Anonymization: {anon_txt}</p>' + df_html(glob)
            + f'<h4>Checks</h4>{flag_html}<h4>Channels, by detected type</h4>' + tables)


def on_run2(b):
    b.disabled = True
    try:
        with out_sec2:
            clear_output()
            if 'header' not in STATE:
                print('Choose a study folder and a participant first.')
                return
        content = sec2_content()
        with out_sec2:
            clear_output()
            display(HTML(content))
        REPORT_BLOCKS['2'] = [content]
        mark_up_to_date('2')
        log_action('2', 'header shown')
    except Exception as e:
        with out_sec2:
            clear_output()
            print(f'Header error: {e!r}')
    finally:
        b.disabled = False


btn_run2.on_click(on_run2)
BADGES['2'] = widgets.HTML(badge_html('2'))
sec2_box = widgets.VBox([
    training_box(
        'Shows how the recording was exported: per channel, the sampling rate, unit, physical range and '
        'hardware filters written in the EDF header (the signal is not read).',
        'Tool 1 inspects the headers of the whole database (one report per dataset and per participant); '
        'tool 1bis anonymizes them.',
        'Is the header anonymized? Are all EEG channels at the same rate? An exported dynamic range of 500 µV or '
        'less clips large slow waves and artefacts. Are the hardware filters the expected ones?'),
    BADGES['2'], btn_run2, out_sec2])

In [ ]:
# ===== Step 3: Scoring & events =====
btn_run3 = widgets.Button(description='Show the scoring and the events', icon='play', button_style='primary',
                          layout=widgets.Layout(width='280px', height='36px'))
out_sec3 = widgets.Output()
# Automatic scoring (GSSC, tool 3bis): optional, its own buttons (a few minutes on CPU)
tb_gssc_mode = widgets.ToggleButtons(options=['One electrode (fast)', 'Several electrodes (consensus)'],
                                     value='One electrode (fast)', style={'button_width': '230px'})
dd_gssc_eeg = widgets.Dropdown(description='EEG electrode:', options=[], style={'description_width': '110px'},
                               layout=widgets.Layout(width='300px'))
sel_gssc_eeg = widgets.SelectMultiple(description='Electrodes:', options=[], rows=4,
                                      style={'description_width': '110px'}, layout=widgets.Layout(width='300px'))
dd_gssc_eog = widgets.Dropdown(options=['L − R bipolar', 'L and R separately', 'None'], value='L − R bipolar',
                               description='EOG:', style={'description_width': '110px'},
                               layout=widgets.Layout(width='300px'))
btn_gssc_read = widgets.Button(description="Read tool 3bis's scoring", icon='folder-open', disabled=True,
                               layout=widgets.Layout(width='230px'))
btn_gssc_run = widgets.Button(description='Score now with GSSC', icon='play', button_style='info',
                              layout=widgets.Layout(width='230px'))
html_gssc = widgets.HTML('')
out_gssc = widgets.Output()
AS = None                       # tools/autoscoring_lib.py, imported on first use (it loads torch + GSSC)
_GSSC = {'infer': None}         # the GSSC model, loaded once per session


def autoscoring():
    global AS
    if AS is None:
        import autoscoring_lib
        AS = autoscoring_lib
    return AS


def _on_gssc_mode(change):
    one = change['new'].startswith('One')
    dd_gssc_eeg.layout.display = '' if one else 'none'
    sel_gssc_eeg.layout.display = 'none' if one else ''


tb_gssc_mode.observe(_on_gssc_mode, names='value')
sel_gssc_eeg.layout.display = 'none' if tb_gssc_mode.value.startswith('One') else ''


def sec3_content():
    """Figure + the hypnogram checks (tool 3's rules) + the events (tool 4's labels). Returns HTML blocks."""
    labels, orig, custom = STATE['hypno_work'], STATE['hypno_orig'], STATE['custom_stages']
    # the night figure of tool 5's per-recording report (macrostructure_lib), with the manual annotations
    fig = ML.plot_night_overview(labels, custom, events=STATE['onsets'], manual=STATE['manual_events'],
                                 title='Hypnogram in use (rescorings included) and scored events',
                                 manual_color=L.MANUAL_COLOR)
    blocks = [png_html(fig_to_png(fig), 'scoring overview')]
    # --- hypnogram (tool 3) ---
    items = []
    hname = STATE['hypno_path'].name if STATE['hypno_path'] else None
    status_txt = {'ok': 'as long as the recording', 'trim': 'one extra score dropped (tool 7 does the same)',
                  'mismatch': '<b style="color:#c62828;">length mismatch: tool 7 skips this participant</b>',
                  'none': 'no hypnogram'}[STATE['hypno_status']]
    items.append(f'File: <code>{hname or "none"}</code>, {status_txt}.')
    mid = EIO.mid_recording_unscored(orig)
    if mid:
        nums = ', '.join(f'#{i + 1}' for i in mid[:15]) + (' …' if len(mid) > 15 else '')
        items.append(f'<span style="color:#e65100;">{len(mid)} unscored "?" epoch(s) in the middle of the night '
                     f'({nums}): tool 3 lists them for review (mid_uncertain_epochs_to_verify.tsv).</span>')
    outside = sorted(set(orig) - set(L.AASM_STAGES) - set(custom) - {'MT', '?'})
    if outside:
        items.append(f'<span style="color:#e65100;">Labels outside W/N1/N2/N3/R: {", ".join(outside)}: harmonize '
                     'them with tool 3, or declare them as custom stages.</span>')
    if custom:
        items.append(f'Custom stages declared (tool 3): {", ".join(custom)}.')
    n_mod = int((STATE['hypno_work'] != orig).sum())
    if n_mod:
        items.append(f'{n_mod} epoch(s) rescored in the explorer.')
    blocks.append('<h4>Hypnogram</h4><ul>' + ''.join(f'<li>{t}</li>' for t in items) + '</ul>')
    # how the labels of the other hypnogram files map to the one in use (e.g. the raw export -> tool 3)
    edf = Path(STATE['edf'])
    for suf in [o for o in dd_hypno.options
                if o not in (NONE, OTHER_FILE, dd_hypno.value) and 'event' not in o.lower()]:
        try:
            other = EIO.load_hypno_labels(edf.with_name(f'{edf.stem}{suf}'))
        except Exception:
            continue
        if not len(orig) or abs(len(other) - len(orig)) > 1:
            continue
        m = min(len(other), len(orig))
        tab = pd.crosstab(pd.Series(other[:m], name=f'label in {suf}'), pd.Series(orig[:m], name='label in use'))
        blocks.append(f'<p>Labels of <code>{edf.stem}{suf}</code> (rows) against the hypnogram in use (columns), '
                      'epoch by epoch: how one scoring was translated into the other.</p>'
                      + tab.to_html(border=0).replace('<table', '<table style="font-size:0.9em;"'))
    # --- events (tool 4) ---
    on = STATE['onsets']
    if on is not None and len(on):
        n_ep = len(labels)
        idx = np.floor(on['onset_s'].astype(float).values / 30).astype(int)
        inside = (idx >= 0) & (idx < n_ep)
        stage = np.where(inside, np.asarray(labels)[np.clip(idx, 0, max(n_ep - 1, 0))], 'outside')
        ev = on.assign(stage=stage)
        by_label = (ev.groupby(['raw_label', 'type', 'mapped']).size().reset_index(name='n')
                    .sort_values('n', ascending=False))
        by_label['tool 4'] = np.where(by_label['mapped'], 'mapped', 'NOT MAPPED (tool 7 ignores it)')
        order = [s for s in L.AASM_STAGES + list(custom) + ['MT', '?', 'outside'] if s in set(stage)]
        order += [s for s in sorted(set(stage)) if s not in order]
        pivot = pd.crosstab(ev['type'], pd.Series(pd.Categorical(ev['stage'], categories=order), name='stage',
                                                  index=ev.index), dropna=False).reset_index()
        n_out = int((~inside).sum())
        warn = (f'<p style="color:#e65100;">{n_out} event(s) start outside the recording.</p>' if n_out else '')
        blocks.append(f"<h4>Scored events ({len(on)}, {STATE['events_source'].upper()} companion)</h4>{warn}"
                      '<p>Raw label &rarr; label in the toolkit vocabulary (tool 4):</p>'
                      + df_html(by_label[['raw_label', 'type', 'tool 4', 'n']])
                      + '<p>Events per sleep stage at their onset:</p>' + df_html(pivot))
    else:
        blocks.append('<h4>Scored events</h4><p>No scored event for this recording.</p>')
    return blocks


def gssc_disk_paths():
    """Where tool 3bis wrote this night's scoring (derivatives/autoscoring_gssc/<subtree>/)."""
    study, edf = Path(STATE['study']), Path(STATE['edf'])
    try:
        sub = edf.parent.relative_to(study)
    except ValueError:
        sub = Path('.')
    d = study / 'derivatives' / 'autoscoring_gssc' / sub
    return d / f"{STATE['fid']}_hypnodensity.tsv", d / f"{STATE['fid']}_scoring_params.json"


def gssc_prepare():
    """Fill the GSSC widgets (electrodes of tool 2's configuration, tool 3bis's choice of C3 / C4) and say
    whether tool 3bis already scored this night."""
    A = autoscoring()
    cfg = STATE.get('cfg')
    opts = A.eeg_options(cfg) if cfg else []
    dd_gssc_eeg.options = opts
    if opts:
        dd_gssc_eeg.value = next((c for c in ('C3', 'C4') if c in opts), opts[0])
    sel_gssc_eeg.options = opts
    sel_gssc_eeg.value = tuple(opts)
    density, params_path = gssc_disk_paths()
    btn_gssc_read.disabled = not density.exists()
    msgs = []
    if not A.GSSC_OK:
        msgs.append(A.GSSC_MSG)
    if cfg is None:
        msgs.append("<span style=\"color:#e65100;\">GSSC needs tool 2's configuration for this participant "
                    '(channel names, reference, EOG), like tool 3bis.</span>')
    if density.exists():
        try:
            p = json.loads(params_path.read_text(encoding='utf-8')) if params_path.exists() else {}
        except Exception:
            p = {}
        msgs.append(f"Tool 3bis scored this night on {p.get('scored_at', '?')} ({p.get('scoring_mode', '?')}, "
                    f"EEG {', '.join(p.get('eeg_used', [])) or '?'}, EOG {', '.join(p.get('eog_used', [])) or 'none'}): "
                    'read it, or score again here.')
    else:
        msgs.append('Tool 3bis has not scored this night yet: score it here (about 1 min on CPU).')
    html_gssc.value = '<br>'.join(msgs)


def show_gssc(stages, probs, source, notes=()):
    """Tool 3bis's figures and agreement for this night, the GSSC layer of the explorer, the report."""
    A = autoscoring()
    fid, custom = STATE['fid'], STATE['custom_stages']
    has_expert = bool(np.isin(STATE['hypno_orig'], A.STAGES).any())
    expert = np.asarray(STATE['hypno_work']) if has_expert else None
    start_dt = A.recording_start(STATE['edf'])
    blocks = [f'<p><b>Automatic scoring</b>: {source}.</p>']
    blocks += [f'<p style="color:#e65100;">⚠ {n}</p>' for n in notes]
    blocks.append(png_html(fig_to_png(A.plot_hypnograms(stages, expert, fid, custom, start_dt)), 'hypnograms'))
    blocks.append(png_html(fig_to_png(A.plot_hypnodensity(probs, fid, expert, start_dt)), 'hypnodensity'))
    res = None
    if expert is not None:
        res, cm, _cm_long = A.compare_scoring(expert, stages)
        blocks.append('<p>Agreement with the hypnogram in use (rescorings included):</p>' + A.metrics_html(res))
        if res['n_compared']:
            blocks.append(png_html(fig_to_png(A.plot_confusion(
                cm, f'{fid}: kappa {res["kappa"]:.2f}, accuracy {res["accuracy"] * 100:.0f} %')), 'confusion'))
        profile = A.non_aasm_profile(expert, stages, probs, custom)
        if len(profile):
            blocks.append(A.NON_AASM_HELP)
            blocks.append(png_html(fig_to_png(A.plot_non_aasm_profile(
                profile, f'{fid}: expert stages outside W/N1/N2/N3/R')), 'non-AASM'))
    else:
        blocks.append('<p>No hypnogram loaded: nothing to compare with.</p>')
    STATE['gssc'] = {'stages': np.asarray(stages).astype(str), 'probs': np.asarray(probs, dtype=float),
                     'source': source, 'result': res}
    with out_gssc:
        clear_output()
        display(HTML(''.join(blocks)))
    STATE['report3_gssc'] = blocks
    REPORT_BLOCKS['3'] = STATE.get('report3_overview', []) + blocks
    box_gssc_nav.layout.display = ''
    render_explorer()
    kappa = f", kappa {res['kappa']:.3f}" if res and res['n_compared'] else ''
    log_action('3', 'automatic scoring', f'{source}{kappa}')


def on_run3(b):
    b.disabled = True
    try:
        with out_sec3:
            clear_output()
            if 'P' not in STATE:
                print('Load the recording first (step 1).')
                return
        blocks = sec3_content()
        with out_sec3:
            clear_output()
            display(HTML(''.join(blocks)))
        STATE['report3_overview'] = blocks
        REPORT_BLOCKS['3'] = blocks + STATE.get('report3_gssc', [])
        mark_up_to_date('3')
        log_action('3', 'scoring and events shown')
        try:
            gssc_prepare()
        except Exception as e:
            html_gssc.value = f'<span style="color:#c62828;">Automatic scoring unavailable: {e!r}</span>'
    except Exception as e:
        with out_sec3:
            clear_output()
            print(f'Step 3 error: {e!r}')
    finally:
        b.disabled = False


def on_gssc_read(b):
    b.disabled = True
    try:
        A = autoscoring()
        density, params_path = gssc_disk_paths()
        dens = pd.read_csv(density, sep='\t')
        p = json.loads(params_path.read_text(encoding='utf-8')) if params_path.exists() else {}
        show_gssc(dens['stage_gssc'].astype(str).values, dens[[f'p_{s}' for s in A.STAGES]].values,
                  f"read from tool 3bis (<code>{density.name}</code>, scored {p.get('scored_at', '?')}, "
                  f"{p.get('scoring_mode', '?')}, EEG {', '.join(p.get('eeg_used', [])) or '?'}, "
                  f"EOG {', '.join(p.get('eog_used', [])) or 'none'})", p.get('notes', []))
    except Exception as e:
        with out_gssc:
            clear_output()
            print(f'Could not read tool 3bis scoring: {e!r}')
    finally:
        b.disabled = False


def on_gssc_run(b):
    b.disabled = True
    try:
        with out_gssc:
            clear_output()
            if 'P' not in STATE:
                print('Load the recording first (step 1).')
                return
        A = autoscoring()
        if not A.GSSC_OK:
            with out_gssc:
                display(HTML(A.GSSC_MSG))
            return
        cfg = STATE.get('cfg')
        if cfg is None:
            with out_gssc:
                print("GSSC needs tool 2's configuration for this participant (channel names, reference, EOG).")
            return
        one = tb_gssc_mode.value.startswith('One')
        wanted = [dd_gssc_eeg.value] if one else list(sel_gssc_eeg.value)
        with out_gssc:
            print('Scoring the night with GSSC (about 1 min on CPU)…')
        if _GSSC['infer'] is None:
            _GSSC['infer'] = A.EEGInfer(use_cuda=A.torch.cuda.is_available())
        stages, _onsets, probs, _winner, n_comb, present, eog_names, notes = A.score_night(
            _GSSC['infer'], STATE['edf'], cfg, wanted, one, dd_gssc_eog.value)
        show_gssc(stages, probs, f"scored here, {'one electrode' if one else 'consensus'} ({', '.join(present)}; "
                                 f"EOG {', '.join(eog_names) or 'none'}; {n_comb} signal combinations)", notes)
    except Exception as e:
        with out_gssc:
            print(f'GSSC failed: {e!r}')
    finally:
        b.disabled = False


btn_run3.on_click(on_run3)
btn_gssc_read.on_click(on_gssc_read)
btn_gssc_run.on_click(on_gssc_run)
BADGES['3'] = widgets.HTML(badge_html('3'))
sec3_box = widgets.VBox([
    training_box(
        'Shows the hypnogram and the scored events of the night, with their labels in the toolkit vocabulary; '
        'optionally scores the night automatically with GSSC and compares it with the hypnogram in use.',
        'Tool 3 harmonizes the hypnogram labels, tool 4 the event labels, tool 3bis scores every recording '
        'automatically (GSSC).',
        'Labels left unmapped, unscored epochs in the middle of the night, events outside the recording, long '
        'stretches where GSSC and the scorer disagree (the explorer jumps from one disagreement to the next).'),
    BADGES['3'], btn_run3, out_sec3,
    widgets.HTML('<h4>Automatic scoring (GSSC, as tool 3bis)</h4>'), html_gssc,
    tb_gssc_mode, dd_gssc_eeg, sel_gssc_eeg, dd_gssc_eog, widgets.HBox([btn_gssc_read, btn_gssc_run]), out_gssc])

In [ ]:
# ===== Step 4: Sleep macrostructure (tool 5's computation, macrostructure_lib.compute_recording) =====
btn_run4 = widgets.Button(description='Compute the macrostructure', icon='play', button_style='primary',
                          layout=widgets.Layout(width='280px', height='36px'))
dd_onset4 = widgets.Dropdown(description='Sleep onset =', options=ML.ONSET_RULES, value=ML.ONSET_RULES[0],
                             style={'description_width': '100px'}, layout=widgets.Layout(width='420px'))
cb_odi4 = widgets.Checkbox(value=False, indent=False, description='ODI (reads the .edf.XML for the depths)',
                           layout=widgets.Layout(width='340px'))
ft_odi_thr4 = widgets.FloatText(description='Desaturation depth >= (%):', value=3.0,
                                style={'description_width': '180px'},
                                layout=widgets.Layout(width='260px', display='' if cb_odi4.value else 'none'))
cb_t90_4 = widgets.Checkbox(value=False, indent=False, description='T90 / T88 (reads the SpO2 signal)',
                            layout=widgets.Layout(width='340px'))
ft_spo2_thr4 = widgets.FloatText(description='SpO2 threshold (%):', value=90.0, style={'description_width': '130px'},
                                 layout=widgets.Layout(width='220px'))
ft_spo2_floor4 = widgets.FloatText(description='Artefact floor (%):', value=50.0,
                                   style={'description_width': '130px'}, layout=widgets.Layout(width='220px'))
box_t90_4 = widgets.HBox([ft_spo2_thr4, ft_spo2_floor4],
                         layout=widgets.Layout(display='' if cb_t90_4.value else 'none'))
cb_odi4.observe(lambda c: setattr(ft_odi_thr4.layout, 'display', '' if c['new'] else 'none'), names='value')
cb_t90_4.observe(lambda c: setattr(box_t90_4.layout, 'display', '' if c['new'] else 'none'), names='value')
out_sec4 = widgets.Output()


def macro_cfg():
    """Tool 5's parameters (its get_params), with tool 5's defaults for what this step does not expose."""
    return dict(epoch_s=ML.DEFAULT_EPOCH_S, lights_source='auto', lights_col='Luminosité', light_ch='',
                lights_off_col='lights_off', lights_on_col='lights_on', onset_rule=dd_onset4.value, events=True,
                odi=bool(cb_odi4.value), odi_thr=float(ft_odi_thr4.value), t90=bool(cb_t90_4.value), spo2_ch='',
                spo2_thr=float(ft_spo2_thr4.value), spo2_floor=float(ft_spo2_floor4.value),
                other_stage_policy=ML.OTHER_STAGE_POLICY, event_count_window=ML.EVENT_COUNT_WINDOW)


def macro_recording():
    """The recording as tool 5's scan describes it (header, hypnogram, summary txt, event XML, Light / SpO2)."""
    edf = Path(STATE['edf'])
    header = ML.read_edf_header_info(edf)
    _txt, _csv, xml = EIO.event_companion_paths(edf, STUDY['txt_suffix'], STUDY['csv_suffix'])
    summ = edf.with_name(f"{edf.stem}{STUDY['summary_suffix']}") if STUDY.get('summary_suffix') else None
    return {'file_id': STATE['fid'], 'edf': edf, 'header': header, 'hypno': STATE['hypno_path'],
            'summary_txt': summ if summ is not None and summ.exists() else None, 'evt_xml': xml,
            'light_ch': ML.find_channel(header['ch_names'], ML.LIGHT_CH_PATTERN),
            'spo2_ch': ML.find_channel(header['ch_names'], ML.SPO2_CH_PATTERN)}


def hypno_suffix_in_use():
    """What tool 5 names the hypnogram by in its messages: the suffix, or the file name of one chosen by hand."""
    if STATE.get('hypno_by_hand'):
        return STATE['hypno_path'].name
    return dd_hypno.value


def macro_subject_row():
    """This participant's row of the study's participant table (tool 5's subject_row), or None."""
    info, col = STUDY.get('subj_info'), STUDY.get('subj_col')
    if info is None or not col or col not in info.columns:
        return None
    hit = info[info[col].astype(str).str.strip().map(ML.norm) == ML.norm(STATE['fid'])]
    return hit.iloc[0].to_dict() if len(hit) else None


def macro_stages():
    """None = tool 5 reads the hypnogram file itself (nothing rescored); otherwise that file with the epochs
    rescored in the explorer replaced, or the scoring done here when there is no hypnogram file."""
    work, orig = STATE['hypno_work'], STATE['hypno_orig']
    if STATE['hypno_path'] is None:
        return np.array(work)
    changed = np.where(work != orig)[0]
    if not len(changed):
        return None
    stages = ML.load_hypnogram(STATE['hypno_path'])
    changed = changed[changed < len(stages)]
    stages[changed] = work[changed]
    return stages


def tool5_saved_values():
    """Tool 5's own table for this participant ({file_id}_sleep_metrics.tsv), when it has run: {metric: value}."""
    study, edf = Path(STATE['study']), Path(STATE['edf'])
    try:
        sub = edf.parent.relative_to(study)
    except ValueError:
        sub = Path('.')
    f = study / 'derivatives' / 'features_macrostructure' / sub / f"{STATE['fid']}_sleep_metrics.tsv"
    if not f.exists():
        return None
    df = pd.read_csv(f, sep='\t')
    return dict(zip(df['metric'].astype(str), df['value']))


def tool5bis_decision_html():
    """Tool 5bis's last validation for this participant (participant_selection.tsv), when it has run."""
    f = Path(STATE['study']) / 'reports_features_macrostructure' / 'participant_selection.tsv'
    if not f.exists():
        return '<p>Tool 5bis has not validated a selection in this study yet.</p>'
    try:
        sel = pd.read_csv(f, sep='\t', dtype={'file_id': str})
        row = sel[sel['file_id'].map(ML.norm) == ML.norm(STATE['fid'])]
        if not len(row):
            return '<p>This participant is not in tool 5bis\'s last selection.</p>'
        r = row.iloc[0]
        if str(r.get('excluded')).lower() == 'true':
            return (f'<p style="color:#b71c1c;">Tool 5bis\'s last validation <b>excludes</b> this participant: '
                    f'{escape(str(r.get("exclude_reason", "")))}.</p>')
        return '<p style="color:#1b5e20;">Tool 5bis\'s last validation keeps this participant.</p>'
    except Exception as e:
        return f'<p style="color:#e65100;">⚠ participant_selection.tsv unreadable: {escape(str(e))}</p>'


def sec4_content(r):
    """Tool 5's per-recording report (macrostructure_lib: the same night figure, metrics, checks, provenance),
    plus the comparison with tool 5's saved table and tool 5bis's decision."""
    metrics = r['metrics']
    fig = ML.plot_night_overview(r['stages'], STATE['custom_stages'],
                                 events=ML.events_for_display(r.get('events'), STUDY.get('event_remap')),
                                 lights=r['lights'], extras=r['extras'],
                                 title='Hypnogram in use and scored events (shaded: time in bed, where the indices '
                                       'count the events)')
    blocks = [png_html(fig_to_png(fig), 'macrostructure')]
    saved = tool5_saved_values()
    ahi = metrics.get('ahi')
    if ahi is not None and np.isfinite(float(ahi)):
        band = next(lbl for lo, hi, lbl in ML.AHI_SEVERITY_BANDS if float(ahi) >= lo and (hi is None or float(ahi) < hi))
        blocks.append(f'<p>AHI {float(ahi):.1f}/h: <b>{band}</b> (AASM severity bands).</p>')
    blocks.append('<h4>Metrics</h4><p><small>'
                  + ("Last column: tool 5's saved table for this participant." if saved is not None else
                     'Tool 5 has not run on this participant yet.') + '</small></p>'
                  + ML.recording_metrics_html(metrics, saved))
    blocks.append('<h4>Checks (tool 5)</h4>' + ML.recording_checks_html(r['checks']))
    blocks.append('<h4>Where the numbers come from</h4>' + ML.recording_provenance_html(r['provenance']))
    blocks.append('<h4>Participant selection (tool 5bis)</h4>' + tool5bis_decision_html())
    return blocks


def on_run4(b):
    b.disabled = True
    try:
        with out_sec4:
            clear_output()
            if 'P' not in STATE:
                print('Load the recording first (step 1).')
                return
            print('Computing the macrostructure…')
        try:
            r = ML.compute_recording(macro_recording(), macro_cfg(), STATE['custom_stages'],
                                     STUDY.get('event_remap') or None, macro_subject_row(), hypno_suffix_in_use(),
                                     STUDY['txt_suffix'], STUDY['csv_suffix'], stages=macro_stages())
        except Exception as e:
            with out_sec4:
                clear_output()
                display(HTML(f'<p style="color:#c62828;"><b>Tool 5 cannot compute this recording</b>: '
                             f'{escape(str(e))} (it would be listed in sleep_metrics_failed.tsv).</p>'))
            return
        STATE['macro'] = r
        blocks = sec4_content(r)
        with out_sec4:
            clear_output()
            display(HTML(''.join(blocks)))
        REPORT_BLOCKS['4'] = blocks
        mark_up_to_date('4')
        m = r['metrics']
        log_action('4', 'macrostructure computed',
                   f"TST {m.get('tst_min', np.nan):.1f} min, SE {m.get('se_pct', np.nan):.1f} %, "
                   f"lights from {r['lights']['source']}")
    except Exception as e:
        with out_sec4:
            clear_output()
            print(f'Step 4 error: {e!r}')
    finally:
        b.disabled = False


btn_run4.on_click(on_run4)
BADGES['4'] = widgets.HTML(badge_html('4'))
sec4_box = widgets.VBox([
    training_box(
        'Computes the classical PSG summary of the night from the hypnogram in use (rescorings included) and the '
        'scored events: time in bed, sleep onset latency, total sleep time, efficiency, WASO, stage durations and '
        'latencies, apnea-hypopnea, limb-movement and arousal indices, optionally ODI and T90.',
        'Tool 5 computes it for every recording (same code), tool 5bis selects the participants on these values.',
        'Values outside the indicative ranges (they depend on age), where lights-off / lights-on come from (the '
        'recording bounds are a last resort), event labels not mapped by tool 4, and whether tool 5bis keeps '
        'this participant.'),
    BADGES['4'], dd_onset4, widgets.HBox([cb_odi4, ft_odi_thr4]), cb_t90_4, box_t90_4, btn_run4, out_sec4])

In [ ]:
# ===== Step 5: Signal quality (tool 6's computation and figures, quality_lib) =====
# Thresholds and options with tool 6's defaults (its Section 2 widgets).
_w5 = dict(style={'description_width': '150px'}, layout=widgets.Layout(width='260px'))
ft_flat5 = widgets.BoundedFloatText(value=3.5, min=0.0, max=100.0, step=0.5, description='flat_pct (%) >', **_w5)
ft_bounds5 = widgets.BoundedFloatText(value=1.0, min=0.0, max=100.0, step=0.1, description='bounds_pct (%) >', **_w5)
it_peaks5 = widgets.BoundedIntText(value=2, min=2, max=10, step=1, description='n_peaks >=', **_w5)
ft_std5 = widgets.BoundedFloatText(value=5.0, min=0.0, max=500.0, step=0.5, description='std (µV) <', **_w5)
ft_hist5 = widgets.BoundedFloatText(value=1.0, min=0.0, max=100.0, step=0.1, description='hist_extreme (%) >', **_w5)
cb_resample5 = widgets.Checkbox(value=False, indent=False, description='Resample before the analysis',
                                layout=widgets.Layout(width='240px'))
it_target5 = widgets.BoundedIntText(value=256, min=1, max=10000, description='to (Hz):', style={'description_width': '60px'},
                                    layout=widgets.Layout(width='160px', display='' if cb_resample5.value else 'none'))
cb_hp5 = widgets.Checkbox(value=False, indent=False, description='High-pass before the analysis',
                          layout=widgets.Layout(width='240px'))
ft_hp5 = widgets.BoundedFloatText(value=0.5, min=0.01, max=5.0, step=0.05, description='at (Hz):',
                                  style={'description_width': '60px'},
                                  layout=widgets.Layout(width='160px', display='' if cb_hp5.value else 'none'))
cb_resample5.observe(lambda c: setattr(it_target5.layout, 'display', '' if c['new'] else 'none'), names='value')
cb_hp5.observe(lambda c: setattr(ft_hp5.layout, 'display', '' if c['new'] else 'none'), names='value')
btn_run5 = widgets.Button(description='Measure the signal quality', icon='play', button_style='primary',
                          layout=widgets.Layout(width='280px', height='36px'))
html_prog5 = widgets.HTML('')
out_sec5 = widgets.Output()
dd_ch5 = widgets.Dropdown(description='Channel:', options=[], style={'description_width': '60px'},
                          layout=widgets.Layout(width='240px'))
btn_ch5_explorer = widgets.Button(description='Show in the explorer', icon='eye', layout=widgets.Layout(width='190px'),
                                  tooltip="The explorer's navigator spectrogram on this channel")
box_ch5 = widgets.HBox([dd_ch5, btn_ch5_explorer])
out_ch5 = widgets.Output()
# closed by default (a tidy tab); its figures are drawn ahead, after the overview, so it opens at once
acc_ch5 = widgets.Accordion(children=[widgets.VBox([box_ch5, out_ch5])], selected_index=None,
                            layout=widgets.Layout(display='none'))
acc_ch5.set_title(0, 'Per-channel inspection')
# columns of tool 6's {file_id}_quality_metrics.tsv shown in the table (the order of the file)
QUALITY_COLUMNS = ['channel', 'std_uV', 'p99_abs_uV', 'p999_abs_uV', 'flat_pct', 'bounds_pct', 'hist_extreme_pct',
                   'n_peaks', 'kurtosis', 'skewness', 'mean_uV']


def quality_thresholds():
    """Tool 6's thresholds dict (its run loop builds the same from its widgets)."""
    return {'flat_pct': float(ft_flat5.value), 'bounds_pct': float(ft_bounds5.value), 'n_peaks': int(it_peaks5.value),
            'std_low': float(ft_std5.value), 'hist_extreme_pct': float(ft_hist5.value)}


def quality_labels():
    """The hypnogram as tool 6 reads it (the whole file: tool 6 applies its own length rule), with the epochs
    rescored in the explorer replaced; the scoring done here when there is no hypnogram file."""
    work, orig = STATE['hypno_work'], STATE['hypno_orig']
    if STATE['hypno_path'] is None:
        return np.array(work)
    labels = EIO.load_hypno_labels(STATE['hypno_path'])
    changed = np.where(work != orig)[0]
    changed = changed[changed < len(labels)]
    labels[changed] = work[changed]
    return labels


def compute_quality():
    """Tool 6's run for this participant (quality_lib, the same calls in the same order): optional resampling
    then high-pass, the hypnogram rules, every channel's metrics and flags, the display limits, and the
    hypnospectrogram of every channel. The explorer's signal is never modified (a copy is filtered)."""
    raw, sf, fid = STATE['raw'], STATE['sf'], STATE['fid']
    notes = []
    do_resample, do_hp = bool(cb_resample5.value), bool(cb_hp5.value)
    target, hp = (int(it_target5.value) if do_resample else None), (float(ft_hp5.value) if do_hp else None)
    if do_resample or do_hp:
        raw = raw.copy()

    def show_status(text):
        html_prog5.value = f'<i>{text}</i>'

    sf = QL.prepare_signal(raw, sf, fid, do_resample, target, do_hp, hp, warn=notes.append, status=show_status)
    # EDF physical range of each channel (bounds_pct), read from the header like tool 6 does
    cfg = STATE['cfg']
    remap = cfg['remap'] if cfg is not None and cfg.get('remap') else {ch: ch for ch in STATE['raw'].ch_names}
    phys_bounds_by_name, final_to_orig = EIO.phys_bounds_for_remap(STATE['edf'], remap)
    # hypnogram: tool 6's label rule, then its length rule
    expert_hypno, hypno_vec, warn, warn_plain = None, None, None, None
    if STATE['hypno_path'] is None and (STATE['hypno_work'] == '?').all():
        warn = 'No hypnogram: no per-stage metrics and no spectrogram (tool 6 needs one for both).'
    else:
        expert_hypno = quality_labels()
        hypno_vec, warn, warn_plain = QL.hypno_vector(expert_hypno, STATE['custom_stages'])
    expert_hypno, hypno_vec, warn, warn_plain = QL.check_hypno_length(raw, sf, expert_hypno, hypno_vec, warn, warn_plain)
    n_ch = len(raw.ch_names)

    def metrics_progress(ci, ch):
        html_prog5.value = f'<i>Metrics: {ch} ({ci + 1}/{n_ch})…</i>'

    def spectrogram_progress(ci, ch, done):
        if not done:
            html_prog5.value = f'<i>Spectrogram: {ch} ({ci + 1}/{n_ch})…</i>'

    ch_metrics, ch_flags, rows, stage_rows = QL.compute_channel_metrics(
        raw, sf, fid, quality_thresholds(), hypno_vec, STATE['custom_stages'], phys_bounds_by_name, final_to_orig,
        on_channel=metrics_progress)
    y_lim_ts, x_lim_hist, hist_y_max = QL.display_limits(ch_metrics, ch_flags)
    ch_sxx, sxx_f, sxx_t = QL.compute_channel_spectrograms(raw, sf, fid, hypno_vec, warn=notes.append,
                                                           on_channel=spectrogram_progress)
    return dict(raw=raw, sf=sf, ch_metrics=ch_metrics, ch_flags=ch_flags, rows=rows, stage_rows=stage_rows,
                y_lim_ts=y_lim_ts, x_lim_hist=x_lim_hist, hist_y_max=hist_y_max, ch_sxx=ch_sxx, sxx_f=sxx_f,
                sxx_t=sxx_t, expert_hypno=expert_hypno, hypno_vec=hypno_vec, hypno_warning=warn, notes=notes,
                thresholds=quality_thresholds(), resampled=target if do_resample else None, highpass=hp)


def tool6_saved_rows():
    """Tool 6's own table for this participant ({file_id}_quality_metrics.tsv), when it has run."""
    study, edf = Path(STATE['study']), Path(STATE['edf'])
    try:
        sub = edf.parent.relative_to(study)
    except ValueError:
        sub = Path('.')
    f = study / 'reports_quality_overview' / sub / f"{STATE['fid']}_quality_metrics.tsv"
    if not f.exists():
        return None
    return pd.read_csv(f, sep='\t', dtype={'file_id': str})


def quality_table_html(rows, saved):
    """One row per channel (tool 6's columns), flagged channels shaded, and when tool 6 has run on this
    participant a last column comparing with its saved table."""
    df = pd.DataFrame(rows)
    cols = [c for c in QUALITY_COLUMNS if c in df.columns]
    head = ''.join(f'<th>{c}</th>' for c in cols) + '<th>flags</th>'
    head += '<th>tool 6 on disk</th>' if saved is not None else ''
    body = ''
    for _, r in df.iterrows():
        cells = ''.join(f'<td>{r[c]}</td>' if c in ('channel', 'n_peaks') else f'<td>{float(r[c]):.3g}</td>' for c in cols)
        cells += f"<td>{escape(r['suspect_reason'])}</td>"
        if saved is not None:
            s = saved[saved['channel'] == r['channel']]
            if not len(s):
                cells += '<td style="color:#e65100;">not in tool 6\'s table</td>'
            else:
                s = s.iloc[0]
                diff = [c for c in cols[1:] if c in s.index and not np.isclose(float(r[c]), float(s[c]), rtol=1e-9, atol=1e-12)]
                if str(s.get('suspect_reason', '') if pd.notna(s.get('suspect_reason')) else '') != r['suspect_reason']:
                    diff.append('flags')
                cells += ('<td style="color:#1b5e20;">✓ same</td>' if not diff else
                          f'<td style="color:#c62828;">≠ {", ".join(diff)}</td>')
        style = ' style="background:#fdecea;"' if r['exclude'] else ''
        body += f'<tr{style}>{cells}</tr>'
    return f'<table style="border-collapse:collapse;font-size:0.85em;"><tr>{head}</tr>{body}</table>'


def quality_flag_html(q):
    """Tool 6's overview box: the suspicious channels and why, or all within the thresholds."""
    flagged = {ch: f for ch, f in q['ch_flags'].items() if f}
    if flagged:
        items = ''.join(f'<li><b>{ch}</b>: {escape(", ".join(f))}</li>' for ch, f in flagged.items())
        html = ('<div style="background:#f8d7da;padding:8px;border-left:4px solid #dc3545;margin-bottom:8px;">'
                f'<b>⚠ Suspicious channels ({len(flagged)}/{len(q["ch_flags"])})</b><ul>{items}</ul></div>')
    else:
        html = ('<div style="background:#d4edda;padding:8px;border-left:4px solid #28a745;margin-bottom:8px;">'
                '<b>✓ All channels within thresholds</b></div>')
    if q['hypno_warning']:
        html += ('<div style="background:#fff3cd;padding:6px;border-left:4px solid #ffc107;margin-bottom:8px;">'
                 f"⚠ {q['hypno_warning']}</div>")
    for n in q['notes']:
        html += f'<p style="color:#e65100;">{escape(n)}</p>'
    steps = [f"resampled to {q['resampled']} Hz" if q['resampled'] else '',
             f"high-passed at {q['highpass']:g} Hz" if q['highpass'] else '']
    steps = [s for s in steps if s]
    html += (f'<p><small>Signal analysed: the EEG as recorded{", " + " and ".join(steps) if steps else ""} '
             f"({q['sf']:g} Hz), no re-reference, like tool 6.</small></p>")
    return html


def stage_table_html(stage_rows):
    """Per-stage metrics (tool 6's {file_id}_quality_by_stage.tsv): std and p99.9 per channel and stage."""
    if not stage_rows:
        return '<p><small>No per-stage metrics (no valid hypnogram).</small></p>'
    df = pd.DataFrame(stage_rows)
    order = [s for s in ['W', 'N1', 'N2', 'N3', 'R'] + list(STATE['custom_stages']) if s in set(df['stage'])]
    parts = []
    for col, label in (('std_uV', 'std (µV)'), ('p999_abs_uV', 'p99.9 |amplitude| (µV)'), ('flat_pct', 'flat (%)')):
        tab = df.pivot(index='channel', columns='stage', values=col).reindex(columns=order)
        tab = tab.reindex([c for c in STATE['raw'].ch_names if c in tab.index]).round(2).reset_index()
        parts.append(f'<p style="margin:6px 0 2px 0;"><b>{label}</b></p>' + df_html(tab))
    return ('<details><summary><b>Per-stage metrics</b> (the 30 s epochs of each stage, concatenated)</summary>'
            + ''.join(parts) + '</details>')


def quality_figures(q):
    """Tool 6's overview figures (the same functions): butterfly time series, PSD, median spectrogram,
    amplitude distributions, metrics averaged across channels. The pooled and per-electrode metric figures are
    drawn by electrode_figures, for the inspection report only."""
    raw, sf = q['raw'], q['sf']
    figs = []
    for title, make in (
            ('All channels: time series (butterfly)', lambda: QL.plot_overlay_timeseries(raw, sf, y_lim=q['y_lim_ts'])),
            ('All channels: PSD', lambda: QL.plot_overlay_psd(raw, sf)),
            ('All channels: amplitude distribution', lambda: QL.plot_overlay_distribution(raw, x_lim=q['x_lim_hist']))):
        try:
            figs.append((title, png_html(fig_to_png(make()), title)))
        except Exception as e:
            figs.append((title, f'<p style="color:#c62828;">⚠ {escape(title)}: {escape(str(e))}</p>'))
    if q['ch_sxx']:
        try:
            n_flag = sum(1 for c in q['ch_sxx'] if q['ch_flags'].get(c))
            fig = QL.plot_median_hypnospectrogram(q['sxx_f'], q['sxx_t'], list(q['ch_sxx'].values()),
                                                  q['expert_hypno'], STATE['custom_stages'])
            fig.set_size_inches(10, 3.5)
            title = f"All channels: spectrogram (median of {len(q['ch_sxx'])} channels, {n_flag} flagged, all kept)"
            figs.insert(1, (title, png_html(fig_to_png(fig), title)))
        except Exception as e:
            figs.append(('median spectrogram', f'<p style="color:#c62828;">⚠ median spectrogram: {escape(str(e))}</p>'))
    avg = QL.build_avg_metrics_html(q['ch_metrics'], raw.ch_names)
    return figs, avg


def electrode_figures(q):
    """Tool 6's pooled and per-electrode metric figures: large, so kept in the inspection report only (folded)."""
    df_sub = pd.DataFrame(q['rows'])
    df_stage = pd.DataFrame(q['stage_rows']) if q['stage_rows'] else None
    figs = []
    for title, make in (
            ('All electrodes: metric distribution', lambda: QL.plot_participant_pooled_boxplots(df_sub, df_stage, STATE['custom_stages'])),
            ('All electrodes: metric by electrode', lambda: QL.plot_participant_electrode_scatter(df_sub, STATE['custom_stages']))):
        try:
            figs.append((title, png_html(fig_to_png(make()), title)))
        except Exception as e:
            figs.append((title, f'<p style="color:#c62828;">⚠ {escape(title)}: {escape(str(e))}</p>'))
    return figs


def channel_metrics_html(ch, q):
    """Tool 6's per-channel metrics table, with the interpretation column (thresholds of this run)."""
    m, flags, th = q['ch_metrics'][ch], q['ch_flags'][ch], q['thresholds']
    interp = dict(QL.METRIC_INTERPRETATIONS)
    interp.update({'Flat signal (%)': f"flag if &gt; {th['flat_pct']:g}%",
                   'At EDF bounds (%)': f"flag if &gt; {th['bounds_pct']:g}%",
                   'Extreme histogram (%)': f"flag if &gt; {th['hist_extreme_pct']:g}%",
                   'Std dev': f"flag if &lt; {th['std_low']:g} µV (dead / flat channel)"})
    rows = [('Mean', f"{m['mean_uV']:.2f} µV"), ('Std dev', f"{m['std_uV']:.2f} µV"),
            ('p99 |amplitude|', f"{m['p99_abs_uV']:.1f} µV"), ('p99.9 |amplitude|', f"{m['p999_abs_uV']:.1f} µV"),
            ('Kurtosis (Fischer)', f"{m['kurtosis']:.2f}"), ('Skewness', f"{m['skewness']:.3f}"),
            ('Flat signal (%)', f"{m['flat_pct']:.2f}%"), ('At EDF bounds (%)', f"{m['bounds_pct']:.3f}%"),
            ('Extreme histogram (%)', f"{m['hist_extreme_pct']:.3f}%"), ('n_peaks (distribution)', str(m['n_peaks']))]
    body = ''.join(f'<tr><td>{a}</td><td style="text-align:right;">{b}</td>'
                   f'<td style="color:#555;font-size:0.85em;">{interp.get(a, "")}</td></tr>' for a, b in rows)
    flag = (f'<div style="background:#f8d7da;padding:4px 8px;"><b>Flags:</b> {escape("; ".join(flags))}</div>' if flags
            else '<div style="background:#f0f0f0;padding:4px 8px;"><b>Flags:</b> none</div>')
    return flag + f'<table style="border-collapse:collapse;font-size:0.9em;">{body}</table>'


def render_channel5(change=None):
    """The figures of one channel, as in tool 6's report: histogram, PSD, time series, spectrogram, metrics.
    Step 5 draws them right after its overview figures, inside the closed per-channel inspection, so that
    opening it is immediate."""
    q = STATE.get('quality')
    if q is None or dd_ch5.value not in q['ch_metrics']:
        return
    ch = dd_ch5.value
    with out_ch5:
        clear_output(wait=True)
        try:
            ci = q['raw'].ch_names.index(ch)
            sig = q['raw'].get_data(picks=[ci])[0] * 1e6
            m, flags = q['ch_metrics'][ch], q['ch_flags'][ch]
            parts = [channel_metrics_html(ch, q)]
            parts.append(png_html(fig_to_png(QL.plot_histogram_figure(
                m, ch, y_max=q['hist_y_max'] if not flags else None, x_lim=q['x_lim_hist'])), 'histogram'))
            parts.append(png_html(fig_to_png(QL.plot_psd_figure(sig, q['sf'])), 'PSD'))
            parts.append(png_html(fig_to_png(QL.plot_timeseries_figure(sig, q['sf'], q['y_lim_ts'])), 'time series'))
            if ch in q['ch_sxx']:
                fig = QL.render_hypnospectrogram(q['sxx_f'], q['sxx_t'], q['ch_sxx'][ch].astype(np.float64),
                                                 q['expert_hypno'], STATE['custom_stages'])
                fig.set_size_inches(10, 3.5)
                parts.append(png_html(fig_to_png(fig), 'spectrogram'))
            else:
                parts.append('<p><small>No spectrogram: no valid hypnogram, or its computation failed.</small></p>')
            display(HTML(''.join(parts)))
        except Exception as e:
            print(f'Channel display error: {e!r}')


def on_ch5_explorer(b):
    if dd_ch5.value in list(dd_nav_ch.options):
        dd_nav_ch.value = dd_ch5.value
        html_explorer_info.value = (f'<i>Navigator spectrogram on <b>{dd_ch5.value}</b> (from step 5).</i>')


def on_run5(b):
    b.disabled = True
    try:
        with out_sec5:
            clear_output()
            if 'P' not in STATE:
                print('Load the recording first (step 1).')
                return
        out_ch5.clear_output()
        acc_ch5.layout.display = 'none'
        html_prog5.value = '<i>Measuring the signal quality…</i>'
        try:
            q = compute_quality()
        except Exception as e:
            html_prog5.value = ''
            with out_sec5:
                display(HTML(f'<p style="color:#c62828;"><b>Tool 6 cannot analyse this recording</b>: '
                             f'{escape(str(e))}</p>'))
            return
        STATE['quality'] = q
        html_prog5.value = '<i>Drawing the figures…</i>'
        figs, avg = quality_figures(q)
        saved = tool6_saved_rows()
        blocks = [quality_flag_html(q),
                  '<h4>Metrics per channel</h4><p><small>Shaded = flagged. '
                  + ("Last column: tool 6's saved table for this participant." if saved is not None
                     else 'Tool 6 has not run on this participant yet.') + '</small></p>'
                  + quality_table_html(q['rows'], saved),
                  stage_table_html(q['stage_rows'])]
        fig_blocks = [f'<h4>{escape(t)}</h4>{h}' for t, h in figs]
        fig_blocks.append('<h4>Metrics averaged across channels</h4>' + avg)
        if cb_report.value:
            fig_blocks.append('<p><small>The metric figures per electrode (distribution, by electrode) are in the '
                              'inspection report.</small></p>')
        else:
            fig_blocks.append('<p><small>The metric figures per electrode are drawn for the inspection report '
                              'only: tick "Keep an inspection report" to get them.</small></p>')
        # the overview first; then, while it is being read, the per-channel inspection (closed) and the
        # report's metric figures are drawn
        with out_sec5:
            clear_output()
            display(HTML(''.join(blocks) + ''.join(fig_blocks)))
        html_prog5.value = '<i>Preparing the per-channel inspection…</i>'
        dd_ch5.unobserve(render_channel5, names='value')
        dd_ch5.options = list(q['raw'].ch_names)
        flagged = [c for c, f in q['ch_flags'].items() if f]
        dd_ch5.value = flagged[0] if flagged else q['raw'].ch_names[0]
        dd_ch5.observe(render_channel5, names='value')
        render_channel5()
        acc_ch5.layout.display = ''
        electrode_block = []
        if cb_report.value:
            html_prog5.value = '<i>Drawing the metric figures for the inspection report…</i>'
            electrode_block = ['<details><summary><b>Metrics per electrode</b> (tool 6\'s figures: metric '
                               'distribution, metric by electrode)</summary>'
                               + ''.join(f'<h4>{escape(t)}</h4>{h}' for t, h in electrode_figures(q)) + '</details>']
        html_prog5.value = ''
        # the tables + the first two overview figures + the metric figures (folded)
        REPORT_BLOCKS['5'] = blocks + fig_blocks[:2] + electrode_block
        mark_up_to_date('5')
        log_action('5', 'signal quality measured',
                   f"{len(flagged)}/{len(q['ch_flags'])} channel(s) flagged"
                   + (f" ({', '.join(flagged)})" if flagged else ''))
    except Exception as e:
        html_prog5.value = ''
        with out_sec5:
            clear_output()
            print(f'Step 5 error: {e!r}')
    finally:
        b.disabled = False


btn_run5.on_click(on_run5)
dd_ch5.observe(render_channel5, names='value')
btn_ch5_explorer.on_click(on_ch5_explorer)
BADGES['5'] = widgets.HTML(badge_html('5'))
sec5_box = widgets.VBox([
    training_box(
        'Measures the quality of each EEG channel over the whole night, before any filtering: amplitude '
        'statistics, flat signal, samples at the EDF physical limits, extreme amplitude bins (clipping), number '
        'of peaks of the amplitude distribution, and the spectrogram under the hypnogram. A channel crossing a '
        'threshold is flagged.',
        'Tool 6 does it for every recording and channel (same code) and builds the dataset overview; tool 7 can '
        'read its table to leave the flagged channels out.',
        'The flagged channels and why; a two-peak amplitude distribution (DC drift) or a comb of regular peaks '
        '(quantization); a dead, saturated or noisy stretch in the spectrogram; a channel whose spectrogram sits '
        'tens of dB below the others (attenuated). Open the per-channel inspection for one channel\'s figures; '
        '"Show in the explorer" puts the navigator on that channel.'),
    BADGES['5'],
    widgets.HTML('<b>Thresholds</b> <small>(tool 6\'s defaults)</small>'),
    widgets.HBox([ft_flat5, ft_bounds5, it_peaks5]), widgets.HBox([ft_std5, ft_hist5]),
    widgets.HBox([cb_resample5, it_target5, cb_hp5, ft_hp5]),
    btn_run5, html_prog5, out_sec5, acc_ch5])

In [ ]:
# ===== Step 6: Preprocessing and flagging (tool 7's computation, preprocessing_lib) =====
# Every parameter with tool 7's default (its Section 2 widgets).
_l6 = dict(style={'description_width': 'initial'})


def _px(px):
    return widgets.Layout(width=f'{px}px')


html_ch6 = widgets.HTML('')
box_ch6 = widgets.Box([], layout=widgets.Layout(flex_flow='row wrap'))
CH6 = {}                                       # channel -> its keep checkbox (filled at load)
btn_ch6_step5 = widgets.Button(description="Untick step 5's flagged channels", layout=_px(260),
                               tooltip='Deselect the channels step 5 flags (tool 7 uses tool 6\'s table on disk)')
cb_resample6 = widgets.Checkbox(value=False, indent=False, description='Resample to', layout=_px(105))
it_resample6 = widgets.BoundedIntText(value=256, min=1, max=10000, description='Hz', layout=_px(120), **_l6)
cb_notch6 = widgets.Checkbox(value=True, indent=False, description='Notch at', layout=_px(85))
ft_notch6 = widgets.BoundedFloatText(value=50.0, min=1.0, max=500.0, description='Hz', layout=_px(115), **_l6)
cb_bp6 = widgets.Checkbox(value=True, indent=False, description='Band-pass', layout=_px(95))
ft_l6 = widgets.BoundedFloatText(value=0.1, min=0.0, max=100.0, step=0.05, description='', layout=_px(75))
ft_h6 = widgets.BoundedFloatText(value=50.0, min=1.0, max=500.0, description='to (Hz)', layout=_px(130), **_l6)
dd_epoch6 = widgets.Dropdown(options=[30, 15, 10, 6, 5], value=30, description='Epoch length (s):', layout=_px(200), **_l6)
cb_amp6 = widgets.Checkbox(value=True, indent=False, description='Amplitude p-p >', layout=_px(130))
AMP6 = {s: widgets.BoundedFloatText(value=v, min=1.0, max=5000.0, step=10.0, description=s, layout=_px(105), **_l6)
        for s, v in (('W', 250.0), ('N1', 250.0), ('N2', 300.0), ('N3', 300.0), ('R', 250.0))}
ft_amp_custom6 = widgets.BoundedFloatText(value=250.0, min=1.0, max=5000.0, step=10.0, description='custom stages',
                                          layout=_px(170), **_l6)
cb_grad6 = widgets.Checkbox(value=True, indent=False, description='Gradient >', layout=_px(100))
ft_grad6 = widgets.BoundedFloatText(value=100.0, min=1.0, max=5000.0, step=10.0, description='µV/sample', layout=_px(160), **_l6)
cb_flat6 = widgets.Checkbox(value=True, indent=False, description='Flat p-p <', layout=_px(100))
ft_flat6 = widgets.BoundedFloatText(value=1.0, min=0.01, max=100.0, step=0.1, description='µV', layout=_px(110), **_l6)
cb_1f6 = widgets.Checkbox(value=True, indent=False, description='1/f fit: R² <', layout=_px(115))
ft_r2_6 = widgets.BoundedFloatText(value=0.95, min=0.0, max=1.0, step=0.01, description='', layout=_px(75))
ft_mae6 = widgets.BoundedFloatText(value=0.15, min=0.0, max=1.0, step=0.01, description='or error >', layout=_px(150), **_l6)
ft_fmin6 = widgets.BoundedFloatText(value=2.0, min=0.1, max=100.0, step=0.5, description='fit', layout=_px(95), **_l6)
ft_fmax6 = widgets.BoundedFloatText(value=45.0, min=1.0, max=200.0, step=0.5, description='to (Hz)', layout=_px(130), **_l6)
cb_ss6 = widgets.Checkbox(value=True, indent=False, description='PSD smoothing: LOWESS', layout=_px(180))
ft_ss6 = widgets.BoundedFloatText(value=2.0, min=0.25, max=10.0, step=0.25, description='Hz', layout=_px(110), **_l6)
cb_ssmed6 = widgets.Checkbox(value=True, indent=False, description='after a median of', layout=_px(140))
ft_ssmed6 = widgets.BoundedFloatText(value=3.0, min=0.25, max=10.0, step=0.25, description='Hz', layout=_px(110), **_l6)
cb_ev6 = widgets.Checkbox(value=False, indent=False, description='Scored events (the epoch holding an event onset)',
                          layout=_px(380))
cb_evman6 = widgets.Checkbox(value=False, indent=False, description='include the manual annotations', layout=_px(250))
box_evtypes6 = widgets.Box([], layout=widgets.Layout(flex_flow='row wrap', display='' if cb_ev6.value else 'none'))
EV6 = {}                                       # canonical event type -> its checkbox (filled at load)
cb_ev6.observe(lambda c: setattr(box_evtypes6.layout, 'display', '' if c['new'] else 'none'), names='value')
btn_run6 = widgets.Button(description='Preprocess and flag', icon='play', button_style='primary',
                          layout=widgets.Layout(width='280px', height='36px'))
html_prog6 = widgets.HTML('')
out_sec6 = widgets.Output()


def prefill_step6():
    """At load: one keep checkbox per EEG channel, pre-filled like tool 7 (the channels tool 6's table on disk
    flags are unticked: deselected, kept in the .fif marked bad, never flagged), and the event types of
    event_remap.json (all ticked, as tool 7)."""
    fid = STATE['fid']
    flagged, source = set(), "nothing deselected (tool 6's quality_summary.tsv not found)"
    qpath = Path(STATE['study']) / 'reports_quality_overview' / 'quality_summary.tsv'
    if qpath.exists():
        try:
            q = pd.read_csv(qpath, sep='\t', dtype={'file_id': str})
            rows = q[q['file_id'].map(os.path.normcase) == os.path.normcase(fid)]
            flagged = set(rows.loc[rows['exclude'].astype(bool), 'channel'].astype(str))
            source = (f"tool 6's <code>quality_summary.tsv</code>: {len(flagged)} flagged channel(s) unticked"
                      if len(rows) else "this participant is not in tool 6's quality_summary.tsv: nothing deselected")
        except Exception as e:
            source = f'⚠ quality_summary.tsv unreadable ({escape(str(e))}): nothing deselected'
    CH6.clear()
    for ch in STATE['raw'].ch_names:
        CH6[ch] = widgets.Checkbox(value=ch not in flagged, description=ch, indent=False, layout=_px(110))
    box_ch6.children = list(CH6.values())
    html_ch6.value = (f'<small>Channels kept (unticked = deselected: in the epochs, marked bad, never flagged). '
                      f'Pre-selection like tool 7: {source}.</small>')
    EV6.clear()
    types = sorted({v for v in (STUDY.get('event_remap') or {}).values() if v})
    for t in types:
        EV6[t] = widgets.Checkbox(value=True, description=t, indent=False, layout=_px(200))
    box_evtypes6.children = (list(EV6.values()) if EV6 else
                             [widgets.HTML('<small>No event_remap.json in config_param/ (tool 4): event flagging '
                                           'is not available.</small>')])


def on_ch6_step5(b):
    q = STATE.get('quality')
    if q is None:
        html_ch6.value += '<br><small style="color:#e65100;">Run step 5 first.</small>'
        return
    for ch, cb in CH6.items():
        if q['ch_flags'].get(ch):
            cb.value = False


def preproc_settings():
    """Tool 7's run settings, read from this tab (the dict it builds from its widgets)."""
    ptp = {s: float(w.value) for s, w in AMP6.items()}
    for cs in STATE['custom_stages']:
        ptp[cs] = float(ft_amp_custom6.value)
    fmin, fmax = float(ft_fmin6.value), float(ft_fmax6.value)
    return dict(do_resample=bool(cb_resample6.value), target_freq=int(it_resample6.value) if cb_resample6.value else None,
                do_notch=bool(cb_notch6.value), notch_freq=float(ft_notch6.value), do_filter=bool(cb_bp6.value),
                l_freq=float(ft_l6.value), h_freq=float(ft_h6.value), epoch_sec=int(dd_epoch6.value), ptp=ptp,
                flat=float(ft_flat6.value), grad=float(ft_grad6.value), mae=float(ft_mae6.value),
                r2=float(ft_r2_6.value), fit_fmin=fmin, fit_fmax=fmax, ss=bool(cb_ss6.value), ss_span=float(ft_ss6.value),
                ss_median=bool(cb_ssmed6.value), ss_median_span=float(ft_ssmed6.value),
                do_amplitude=bool(cb_amp6.value), do_gradient=bool(cb_grad6.value), do_flat=bool(cb_flat6.value),
                do_1f=bool(cb_1f6.value), do_event=bool(cb_ev6.value), include_manual=bool(cb_evman6.value),
                event_types=set(t for t, cb in EV6.items() if cb.value))


def step6_thresholds(s):
    """The thresholds in the sidecar format tool 8 reads (its montage colours the traces with them)."""
    return {'amplitude_ptp_uV': dict(s['ptp']), 'flat_ptp_uV': s['flat'], 'gradient_uV_per_sample': s['grad'],
            '1f_mae_max': s['mae'], '1f_r2_min': s['r2']}


def compute_preprocessing():
    """Tool 7's run for this participant (preprocessing_lib, the same calls in the same order) on a copy of
    the recording: channel selection, resampling / re-reference / notch / band-pass, tool 7's hypnogram rules on
    the hypnogram in use, epochs, the deselected channels set aside, PSD (+ smoothing) for the 1/f fit, events,
    every flagging mask, the per-epoch metadata. A fatal step raises PL.StepFailed, as in tool 7."""
    s = preproc_settings()
    fid, edf = STATE['fid'], Path(STATE['edf'])
    notes = []
    if not (s['do_amplitude'] or s['do_flat'] or s['do_gradient'] or s['do_1f'] or s['do_event']):
        raise PL.StepFailed('no method', 'No flagging method ticked: tick at least one.')
    if s['fit_fmax'] <= s['fit_fmin']:
        notes.append(f"⚠ 1/f fit range invalid ({s['fit_fmin']}–{s['fit_fmax']} Hz); falling back to 2–45 Hz.")
        s['fit_fmin'], s['fit_fmax'] = 2.0, 45.0
    event_remap = STUDY.get('event_remap') or {}
    if s['do_event'] and not s['event_types']:
        notes.append('⚠ Event-based flagging is on but no event type selected: disabled for this run.')
        s['do_event'] = False
    if s['do_event'] and not event_remap:
        notes.append('⚠ Event-based flagging is on but no event_remap.json: disabled for this run.')
        s['do_event'] = False

    def show(text):
        html_prog6.value = f'<i>{escape(text)}</i>'

    raw = STATE['raw'].copy()
    selected = [ch for ch, cb in CH6.items() if cb.value]
    deselected = [ch for ch, cb in CH6.items() if not cb.value]
    if not PL.select_montage_channels(raw, selected, deselected):
        raise PL.StepFailed('no channels after rename', 'No channel ticked: nothing to process.')
    show('preprocessing…')
    PL.preprocess_raw(raw, fid, STATE['cfg'] or {}, s['do_resample'], s['target_freq'], s['do_notch'],
                      s['notch_freq'], s['do_filter'], s['l_freq'], s['h_freq'], warn=notes.append, status=show)
    sf = raw.info['sfreq']
    if STATE['hypno_path'] is None and (np.asarray(STATE['hypno_work']) == '?').all():
        raise PL.StepFailed('hypno not found', 'No hypnogram: tool 7 skips this recording (it needs one).')
    labels = np.asarray(quality_labels()).astype('<U10')       # the file whole + rescorings, as tool 7 reads it
    epoch_sec = s['epoch_sec']
    hypno_sub, stage_mapping, hypno_vec = PL.hypnogram_epochs(labels, raw, sf, 30 // epoch_sec,
                                                              STATE['custom_stages'], fid, warn=notes.append)
    show('epoching…')
    epochs, n_epochs = PL.make_epochs(raw, epoch_sec, hypno_sub, hypno_vec, stage_mapping, fid)
    del raw
    bad_names, bad_epochs, _order = PL.set_bads_aside(epochs)
    hypno_epochs = hypno_sub[:n_epochs]
    show('computing PSD / 1f…')
    psds, freqs = PL.epoch_psd_for_1f(epochs, sf, epoch_sec, s['do_1f'], s['fit_fmax'], s['ss'], s['ss_median'],
                                      s['ss_median_span'], s['ss_span'], fid, warn=notes.append)
    event_epoch_mask = event_type_masks = event_counts = None
    if s['do_event']:
        try:
            events_df, ev_src = EIO.load_events(edf, STUDY['txt_suffix'], STUDY['csv_suffix'],
                                                include_manual=s['include_manual'])
            if events_df is None:
                notes.append('⚠ No event companion found: event-based flagging skipped for this recording.')
            else:
                file_remap = EIO.augment_remap_for_manual(event_remap, events_df)
                event_epoch_mask, event_type_masks, event_counts, _onsets = PL.event_flags(
                    events_df, file_remap, s['event_types'], n_epochs, epoch_sec)
        except Exception as e:
            notes.append(f'⚠ Event flagging error: {e}, skipped for this recording.')
            event_epoch_mask = event_type_masks = event_counts = None

    def flag_progress(done, total, msg=''):
        html_prog6.value = f'<i>flagging · {escape(msg)} ({done}/{total})…</i>'

    mask_dict = PL.compute_rejection_masks(
        epochs, sf, hypno_epochs, s['ptp'], s['flat'], s['grad'], freqs, psds, s['mae'], s['r2'],
        event_epoch_mask=event_epoch_mask, fit_fmin=s['fit_fmin'], fit_fmax=s['fit_fmax'], progress=flag_progress,
        do_amplitude=s['do_amplitude'], do_flat=s['do_flat'], do_gradient=s['do_gradient'])
    combined = PL.build_combined_method_matrix(mask_dict)
    reject_epoch, metadata_df = PL.epoch_metadata(mask_dict, hypno_epochs, event_type_masks, n_epochs)
    epochs.metadata = metadata_df
    ch_names = list(epochs.ch_names)
    return dict(settings=s, epochs=epochs, ch_names=ch_names, bad_names=bad_names, mask_dict=mask_dict,
                combined=combined, hypno_epochs=hypno_epochs, n_epochs=n_epochs, epoch_sec=epoch_sec, sf=sf,
                reject_epoch=reject_epoch, event_type_masks=event_type_masks, event_counts=event_counts, notes=notes,
                thresholds=step6_thresholds(s),
                log=PL.build_epoch_channel_rejection_log(mask_dict, hypno_epochs, ch_names, fid),
                summary=PL.build_rejection_summary(mask_dict, hypno_epochs, fid, ch_names,
                                                   custom_stages=STATE['custom_stages']))


def tool7_saved_log():
    """Tool 7's {file_id}_epoch_channel_rejection.tsv for this participant, when it has run."""
    hits = list((Path(STATE['study']) / 'reports_preproc').rglob(f"{STATE['fid']}_epoch_channel_rejection.tsv"))
    return pd.read_csv(hits[0], sep='\t', dtype={'file_id': str}) if hits else None


def tool7_comparison_html(log):
    saved = tool7_saved_log()
    if saved is None:
        return "<p><small>Tool 7 has not run on this participant yet.</small></p>"
    try:
        same = (list(saved.columns) == list(log.columns) and len(saved) == len(log)
                and all((saved[c].astype(str).values == log[c].astype(str).values).all() for c in log.columns))
    except Exception:
        same = False
    if same:
        return (f'<p style="color:#1b5e20;">✓ Identical to tool 7\'s table on disk ({len(log)} epoch × channel '
                'pairs, every flag).</p>')
    return ('<p style="color:#c62828;">≠ Differs from tool 7\'s table on disk: tool 7 ran with other settings '
            '(channels, filters, methods, thresholds, epoch length) or on another hypnogram.</p>')


def sec6_content(r):
    s, mask_dict, combined = r['settings'], r['mask_dict'], r['combined']
    n_ep = r['n_epochs']
    pct_pairs = 100.0 * (combined > 0).mean() if combined.size else 0.0
    pct_ep = 100.0 * int(r['reject_epoch'].sum()) / n_ep if n_ep else 0.0
    blocks = [f'<p><b>Total epochs:</b> {n_ep} of {r["epoch_sec"]} s &nbsp;·&nbsp; <b>Flagged (epoch × channel):</b> '
              f'{pct_pairs:.1f}% &nbsp;·&nbsp; <b>Epochs with ≥ 1 flagged channel:</b> {pct_ep:.1f}% '
              '<small>(flags, not the decision: step 7 decides)</small></p>']
    if r['bad_names']:
        blocks.append(f'<p><b>Deselected channels</b> (in the epochs, marked bad, not flagged): '
                      f'{escape(", ".join(r["bad_names"]))}</p>')
    for n in r['notes']:
        blocks.append(f'<p style="color:#e65100;">{escape(n)}</p>')
    blocks.append(tool7_comparison_html(r['log']))
    try:
        fig = PL.plot_rejection_heatmap(combined, r['ch_names'], r['hypno_epochs'], STATE['fid'],
                                        custom_stages=STATE['custom_stages'])
        blocks.append(png_html(fig_to_png(fig), 'flagging heatmap'))
    except Exception as e:
        blocks.append(f'<p style="color:#c62828;">⚠ heatmap: {escape(str(e))}</p>')
    blocks.append('<h4>Per sleep stage and method</h4><p><small>% (n) of the epoch × channel pairs of each stage '
                  'that are flagged.</small></p>' + PL.build_stage_method_html(r['summary']))
    blocks.append('<h4>Per channel and stage</h4><p><small>% (n) of the epochs of each stage where the channel is '
                  'flagged by any method.</small></p>'
                  + PL.build_channel_stage_html(mask_dict, r['hypno_epochs'], r['ch_names'],
                                                custom_stages=STATE['custom_stages']))
    if s['do_event']:
        ev_html = PL.build_event_type_html(r['event_type_masks'], r['hypno_epochs'], custom_stages=STATE['custom_stages'])
        blocks.append('<h4>Event-based flagging by type</h4>'
                      + (ev_html or '<p><em>Event flagging was on but no event companion was found.</em></p>'))
    steps = [f"resampled to {s['target_freq']} Hz" if s['do_resample'] else 'no resampling',
             f"notch {s['notch_freq']:g} Hz" if s['do_notch'] else 'no notch',
             f"band-pass {s['l_freq']:g}-{s['h_freq']:g} Hz" if s['do_filter'] else 'no band-pass',
             'PSD smoothing ' + ((f"median {s['ss_median_span']:g} Hz + " if s['ss_median'] else '')
                                 + f"LOWESS {s['ss_span']:g} Hz" if s['ss'] else 'off')]
    blocks.append(f'<p><small>Signal: re-referenced as tool 2 declares, {", ".join(steps)}. Methods: '
                  f'{", ".join(PL.METHOD_LABEL[m] for m in PL.METHOD_ORDER if m in mask_dict)}.</small></p>')
    return blocks


def on_run6(b):
    b.disabled = True
    try:
        with out_sec6:
            clear_output()
            if 'P' not in STATE:
                print('Load the recording first (step 1).')
                return
        html_prog6.value = '<i>Preprocessing…</i>'
        try:
            r = compute_preprocessing()
        except PL.StepFailed as e:
            html_prog6.value = ''
            with out_sec6:
                display(HTML(f'<p style="color:#c62828;"><b>Tool 7 would skip this recording</b>: {escape(e.message)} '
                             f'(listed in preprocessing_failed.tsv as "{escape(e.reason)}").</p>'))
            return
        STATE['preproc'] = r
        html_prog6.value = '<i>Drawing…</i>'
        blocks = sec6_content(r)
        html_prog6.value = ''
        with out_sec6:
            clear_output()
            display(HTML(''.join(blocks)))
        REPORT_BLOCKS['6'] = blocks
        mark_up_to_date('6')
        prefill_step7()                        # new flags: step 7's choices again, its former decision dropped
        explorer_after_step6()
        log_action('6', 'preprocessed and flagged',
                   f"{100.0 * (r['combined'] > 0).mean():.1f}% of epoch x channel pairs flagged, epoch {r['epoch_sec']} s")
    except Exception as e:
        html_prog6.value = ''
        with out_sec6:
            clear_output()
            print(f'Step 6 error: {e!r}')
    finally:
        b.disabled = False


btn_run6.on_click(on_run6)
btn_ch6_step5.on_click(on_ch6_step5)
BADGES['6'] = widgets.HTML(badge_html('6'))
_ind = widgets.Layout(margin='0 0 0 24px')
sec6_box = widgets.VBox([
    training_box(
        'Preprocesses the EEG exactly like tool 7 (re-reference declared in tool 2, optional resampling, notch, '
        'band-pass, epochs of 30 s or shorter) and flags each (epoch, channel) pair with every method: amplitude '
        'per stage, gradient, flat signal, 1/f fit quality, and optionally the scored events. Flags are not a '
        'decision: step 7 turns them into keep / reject.',
        'Tool 7 does it for every recording and writes the epochs (derivatives/preproc_epo/) that tools 8, 8bis and '
        '9 read, with these flag tables (reports_preproc/).',
        'Which methods flag, on which channels, in which stages (an electrode flagged everywhere is probably bad); '
        'the heatmap for stretches of artefact. Then look at flagged epochs in the explorer: the "preprocessed" '
        'signal shows them as tool 8 does.'),
    BADGES['6'],
    widgets.HTML('<b>Channels</b>'), html_ch6, box_ch6, btn_ch6_step5,
    widgets.HTML('<b>Preprocessing</b> <small>(tool 7\'s defaults)</small>'),
    widgets.HBox([cb_resample6, it_resample6, cb_notch6, ft_notch6, cb_bp6, ft_l6, ft_h6, dd_epoch6]),
    widgets.HTML('<b>Flagging methods</b>'),
    widgets.HBox([cb_amp6] + list(AMP6.values()) + [ft_amp_custom6]),
    widgets.HBox([cb_grad6, ft_grad6, cb_flat6, ft_flat6]),
    widgets.HBox([cb_1f6, ft_r2_6, ft_mae6, ft_fmin6, ft_fmax6]),
    widgets.HBox([cb_ss6, ft_ss6, cb_ssmed6, ft_ssmed6], layout=_ind),
    widgets.HBox([cb_ev6, cb_evman6]), box_evtypes6,
    btn_run6, html_prog6, out_sec6])

In [ ]:
# ===== Step 7: Rejection decision (tool 8bis's rule, autoreject_lib; or tool 8 / 8bis's tables on disk) =====
dd_src7 = widgets.Dropdown(description='Source:', options=[], style={'description_width': '60px'},
                           layout=widgets.Layout(width='620px'))
html_src7 = widgets.HTML('')
_w7 = dict(style={'description_width': '150px'}, layout=widgets.Layout(width='250px'))
ft_ch7 = widgets.BoundedFloatText(value=AR.DEFAULT_CHANNEL_REJECT_PCT, min=0.0, max=100.0, step=1.0,
                                  description='Channel reject > (%)', **_w7)
ft_ep7 = widgets.BoundedFloatText(value=AR.DEFAULT_EPOCH_REJECT_PCT, min=0.0, max=100.0, step=1.0,
                                  description='Epoch reject > (%)', **_w7)
MT7, ST7, EV7 = {}, {}, {}               # method / reference-stage checkboxes, event rows (checkbox, threshold)
box_methods7 = widgets.Box([], layout=widgets.Layout(flex_flow='row wrap'))
box_events7 = widgets.VBox([], layout=widgets.Layout(margin='2px 0 0 25px'))
box_stages7 = widgets.Box([], layout=widgets.Layout(flex_flow='row wrap'))
box_live7 = widgets.VBox([
    widgets.HBox([ft_ch7, ft_ep7]),
    widgets.HTML('<b>Flagging methods</b> that may reject (all ticked = every flag of step 6, as tool 8bis by '
                 'default):'), box_methods7, box_events7,
    widgets.HTML('<b>Reference stages</b>: they judge the channels (and the exclusion); the epoch rule then '
                 'decides <b>every</b> epoch, whatever its stage.'), box_stages7])
btn_run7 = widgets.Button(description='Decide', icon='play', button_style='primary',
                          layout=widgets.Layout(width='220px', height='36px'))
out_sec7 = widgets.Output()
LIVE7 = 'live'


def decision_tables_on_disk():
    """Tool 8's and tool 8bis's decision tables of this participant ({file_id}_epoch_decision.tsv +
    _channel_decision.tsv, anywhere under derivatives/): [(label, epoch table path, channel table path)]."""
    found = []
    for f in sorted((Path(STATE['study']) / 'derivatives').rglob(f"{STATE['fid']}_epoch_decision.tsv")):
        ch = f.with_name(f"{STATE['fid']}_channel_decision.tsv")
        parts = {p.lower() for p in f.parts}
        tool = ('tool 8 (manual)' if 'rejection_manual' in parts else
                'tool 8bis (automatic)' if 'rejection_auto' in parts else 'decision tables')
        rel = f.parent.relative_to(Path(STATE['study']))
        found.append((f'{tool}: tables in {rel.as_posix()}/', f, ch if ch.exists() else None))
    return found


def prefill_step7():
    """After step 6: the sources (the live rule + the tables on disk that match step 6's epochs), and the
    rule's widgets filled like tool 8bis's scan (methods present, stages, event types)."""
    r6 = STATE.get('preproc')
    STATE.pop('decision', None)
    MT7.clear(); ST7.clear(); EV7.clear()
    box_methods7.children = box_events7.children = box_stages7.children = ()
    if r6 is None:
        dd_src7.options = []
        html_src7.value = '<i>Run step 6 first: the decision is taken on its flags.</i>'
        box_live7.layout.display = 'none'
        return
    opts = [("live: tool 8bis's rule on step 6's flags", LIVE7)]
    notes = []
    for label, ep_path, ch_path in decision_tables_on_disk():
        try:
            n = len(pd.read_csv(ep_path, sep='\t', usecols=['epoch_idx']))
        except Exception as e:
            notes.append(f'{escape(label)}: unreadable ({escape(str(e))})')
            continue
        if n != r6['n_epochs'] or ch_path is None:
            why = f'{n} epochs, step 6 has {r6["n_epochs"]}' if n != r6['n_epochs'] else 'no channel table'
            notes.append(f'{escape(label)}: not offered ({why})')
            continue
        opts.append((label, str(ep_path)))
    dd_src7.options = opts
    dd_src7.value = LIVE7
    html_src7.value = ('<small>' + ('<br>'.join(notes) if notes else
                                    ('Tables of tool 8 / 8bis found on disk: pick one to look at that decision.'
                                     if len(opts) > 1 else 'No decision table of tool 8 / 8bis on disk for this '
                                     'participant.')) + '</small>')
    # tool 8bis's scan, for this participant: methods present, stages, event types (+ this file's counts)
    log = r6['log']
    for m in [m for m in L.METHOD_ORDER if ('flag_' + m) in log.columns]:
        MT7[m] = widgets.Checkbox(value=True, description=L.METHOD_LABEL.get(m, m), indent=False,
                                  layout=widgets.Layout(width='120px'))
    box_methods7.children = tuple(MT7.values())
    ordered = AR.ordered_stages(set(log['stage'].astype(str).unique()))
    ref = AR.default_reference_stages(ordered)
    for st in ordered:
        ST7[st] = widgets.Checkbox(value=(st in ref), description=st, indent=False, layout=widgets.Layout(width='75px'))
    box_stages7.children = tuple(ST7.values())
    flags_df, counts = step6_event_tables()
    types = set(counts)
    if flags_df is not None:
        types |= {c[len('evt_'):] for c in flags_df.columns if c.startswith('evt_')}
    rows = []
    for t in sorted(types):
        cb = widgets.Checkbox(value=True, description=t, indent=False, layout=widgets.Layout(width='190px'))
        thr = widgets.IntText(value=0, layout=widgets.Layout(width='80px'))
        EV7[t] = (cb, thr)
        rows.append(widgets.HBox([cb, widgets.HTML('exclude the participant if &gt;'), thr,
                                  widgets.HTML(f'events (0 = off) <small style="color:#666;">this recording: '
                                               f'{counts.get(t, 0)}</small>')]))
    box_events7.children = tuple(rows)
    ev = MT7.get('event')
    box_events7.layout.display = '' if (ev is not None and ev.value) else 'none'
    if ev is not None:
        ev.observe(lambda c: setattr(box_events7.layout, 'display', '' if c['new'] else 'none'), names='value')
    box_live7.layout.display = ''


def step6_event_tables():
    """Step 6's event sidecars, as tool 7 writes them: the per-epoch event-type flags (only when event flagging
    ran on a file that has events) and the raw count per type ({} when there is none)."""
    r6 = STATE['preproc']
    flags_df = (PL.build_event_epoch_flags(r6['event_type_masks'], r6['hypno_epochs'], STATE['fid'])
                if r6.get('event_type_masks') else None)
    ec = r6.get('event_counts')
    counts = AR.event_counts_from_table(ec) if ec is not None and not ec.empty else {}
    return flags_df, counts


def _on_src7(change):
    box_live7.layout.display = '' if change['new'] == LIVE7 else 'none'


dd_src7.observe(_on_src7, names='value')


def decide_live():
    """Tool 8bis's run for this participant (autoreject_lib, the same calls in the same order) on step 6's flag
    table and event sidecars: the pair matrix of the ticked methods, the channel-first then epoch rule, the
    event-count exclusion, the registry reason, the epoch / channel tables and the decision record."""
    r6, fid = STATE['preproc'], STATE['fid']
    methods = [m for m, cb in MT7.items() if cb.value]
    if not methods:
        raise ValueError('tick at least one flagging method')
    ref = [st for st, cb in ST7.items() if cb.value]
    if not ref:
        raise ValueError('tick at least one reference stage')
    ch_pct, ep_pct = float(ft_ch7.value) / 100.0, float(ft_ep7.value) / 100.0
    event_master_on = 'event' in methods
    event_types = [t for t, (cb, thr) in EV7.items() if cb.value]
    event_thresholds = {t: int(thr.value) for t, (cb, thr) in EV7.items() if cb.value and int(thr.value) > 0}
    flags_df, counts = step6_event_tables()
    piv, stage_by_epoch, meta = AR.pair_matrix_from_tables(r6['log'], methods=methods, event_master=event_master_on,
                                                           event_types=event_types, event_flags=flags_df)
    stages_present = set(stage_by_epoch.dropna().astype(str).unique())
    soi_p = [s for s in ref if s in stages_present]
    if not soi_p:
        raise ValueError('none of the reference stages is in this recording (tool 8bis would list it as failed)')
    dec = AR.auto_reject_decision(piv, stage_by_epoch, soi_p, ch_pct, ep_pct)
    excl_reason = ''
    if event_master_on and event_thresholds:
        excl_reason = AR.event_exclusion_reason(counts, event_thresholds)
    methods_used = '+'.join(meta['used_methods']) or '(none)'
    event_types_used = '+'.join(meta['used_event_types'])
    epoch_df = AR.build_epoch_decision(fid, dec, piv)
    return dict(source=LIVE7, source_label="tool 8bis's rule (live)", piv=piv, dec=dec, meta=meta,
                ref=soi_p, ch_pct=ch_pct, ep_pct=ep_pct, methods_used=methods_used, event_types_used=event_types_used,
                exclusion=AR.registry_reason(excl_reason, dec), event_exclusion=excl_reason,
                epoch_df=epoch_df, channel_df=AR.build_channel_decision(fid, dec, ch_pct, r6['bad_names']),
                record=AR.decision_summary_row(fid, dec, soi_p, ch_pct, ep_pct, methods_used=methods_used,
                                               event_types_used=event_types_used, excluded=bool(excl_reason),
                                               exclude_reason=excl_reason),
                rejected=epoch_df['rejected'].astype(bool).values,
                reasons=epoch_df['reject_reason'].fillna('').astype(str).values,
                dropped=[c for c in dec['rejected_channels']] + [c for c in r6['bad_names']
                                                                 if c not in dec['channels']])


def decision_from_disk(ep_path):
    """A decision of tool 8 / 8bis read from its tables (the epoch count matches step 6's), shown with step 6's
    flags."""
    r6 = STATE['preproc']
    epoch_df = pd.read_csv(ep_path, sep='\t', dtype={'file_id': str}).sort_values('epoch_idx').reset_index(drop=True)
    channel_df = pd.read_csv(Path(ep_path).with_name(f"{STATE['fid']}_channel_decision.tsv"), sep='\t',
                             dtype={'file_id': str})
    label = dict((v, k) for k, v in dd_src7.options)[str(ep_path)]
    dropped = channel_df.loc[channel_df['dropped'].astype(bool), 'channel'].astype(str).tolist()
    piv, stage_by_epoch, meta = AR.pair_matrix_from_tables(r6['log'])       # step 6's flags, every method
    rejected = epoch_df['rejected'].astype(bool).values
    dec = {'channels': list(piv.columns), 'rejected_channels': [c for c in dropped if c in piv.columns],
           'stage_by_epoch': epoch_df.set_index('epoch_idx')['stage'].astype(str),
           'reject_epoch': pd.Series(rejected, index=epoch_df['epoch_idx'].values)}
    return dict(source=str(ep_path), source_label=label, piv=piv, dec=dec, meta=meta, epoch_df=epoch_df,
                channel_df=channel_df, rejected=rejected,
                reasons=epoch_df['reject_reason'].fillna('').astype(str).values if 'reject_reason' in epoch_df else
                np.array([''] * len(epoch_df)), dropped=dropped, exclusion='', event_exclusion='')


def tool8bis_comparison_html(d):
    """The live decision against tool 8bis's tables on disk, when tool 8bis has run on this participant."""
    hits = [ep for label, ep, ch in decision_tables_on_disk() if label.startswith('tool 8bis') and ch is not None]
    if not hits:
        return "<p><small>Tool 8bis has not run on this participant yet.</small></p>"
    try:
        ep = pd.read_csv(hits[0], sep='\t', dtype={'file_id': str})
        ch = pd.read_csv(hits[0].with_name(f"{STATE['fid']}_channel_decision.tsv"), sep='\t', dtype={'file_id': str})
        same_ep = (len(ep) == len(d['epoch_df'])
                   and (ep['rejected'].astype(bool).values == d['rejected']).all())
        same_ch = (sorted(ch.loc[ch['dropped'].astype(bool), 'channel'].astype(str)) == sorted(d['dropped']))
    except Exception:
        same_ep = same_ch = False
    if same_ep and same_ch:
        return ("<p style=\"color:#1b5e20;\">✓ Same decision as tool 8bis's tables on disk (every epoch, every "
                'dropped channel).</p>')
    return ("<p style=\"color:#c62828;\">≠ Differs from tool 8bis's tables on disk: tool 8bis ran with other "
            'settings (methods, thresholds, reference stages), or tool 7 with other settings than step 6.</p>')


def sec7_content(d):
    dec, fid = d['dec'], STATE['fid']
    n, n_rej = len(d['rejected']), int(d['rejected'].sum())
    blocks = [f"<p><b>Source:</b> {escape(d['source_label'])} &nbsp;·&nbsp; <b>Epochs rejected:</b> {n_rej} / {n} "
              f"({100.0 * n_rej / n if n else 0:.1f}%) &nbsp;·&nbsp; <b>Channels dropped:</b> "
              f"{escape(', '.join(d['dropped'])) or 'none'}</p>"]
    if d['source'] == LIVE7:
        if d['exclusion']:
            blocks.append('<p style="background:#ffebee;color:#b71c1c;padding:4px 8px;"><b>Tool 8bis would exclude '
                          f"this participant</b>: {escape(d['exclusion'])} (registry, source 8bis_auto). Tool 0 writes "
                          'nothing in the registry.</p>')
        else:
            blocks.append('<p style="color:#1b5e20;">Tool 8bis would not exclude this participant.</p>')
        if d['meta'].get('warn'):
            blocks.append(f"<p style=\"color:#e65100;\">⚠ {escape(d['meta']['warn'])}</p>")
        blocks.append(tool8bis_comparison_html(d))
        blocks.append(AR.decision_overview_html(fid, dec, d['ref'], d['ch_pct'], d['ep_pct'],
                                                methods_used=d['methods_used'], event_types_used=d['event_types_used'],
                                                exclude_reason=d['event_exclusion']))
    else:
        blocks.append('<p><small>Read from the tables on disk (the flags in the figure are step 6\'s).</small></p>')
        blocks.append('<h4>Rejected epochs per stage</h4>' + AR.per_stage_table_html(dec, []))
        drop = d['channel_df'][d['channel_df']['dropped'].astype(bool)]
        if len(drop):
            blocks.append('<h4>Dropped channels</h4>' + df_html(drop[['channel', 'drop_reason', 'badness_pct']]))
    try:
        custom = [s for s in set(dec['stage_by_epoch'].astype(str)) if s not in AR.AASM_STAGES]
        fig = AR.plot_decision_heatmap(d['piv'], dec, fid, custom)
        blocks.append(png_html(fig_to_png(fig), 'decision heatmap'))
    except Exception as e:
        blocks.append(f'<p style="color:#c62828;">⚠ decision heatmap: {escape(str(e))}</p>')
    return blocks


def on_run7(b):
    b.disabled = True
    try:
        with out_sec7:
            clear_output()
            if STATE.get('preproc') is None:
                print('Run step 6 first: the decision is taken on its flags.')
                return
        try:
            d = decide_live() if dd_src7.value == LIVE7 else decision_from_disk(dd_src7.value)
        except Exception as e:
            with out_sec7:
                display(HTML(f'<p style="color:#c62828;"><b>No decision</b>: {escape(str(e))}</p>'))
            return
        STATE['decision'] = d
        blocks = sec7_content(d)
        with out_sec7:
            clear_output()
            display(HTML(''.join(blocks)))
        REPORT_BLOCKS['7'] = blocks
        mark_up_to_date('7')
        explorer_after_step7()
        log_action('7', 'decision', f"{d['source_label']}: {int(d['rejected'].sum())}/{len(d['rejected'])} epochs "
                                    f"rejected, dropped: {', '.join(d['dropped']) or 'none'}"
                                    + (f"; would be excluded: {d['exclusion']}" if d['exclusion'] else ''))
    except Exception as e:
        with out_sec7:
            clear_output()
            print(f'Step 7 error: {e!r}')
    finally:
        b.disabled = False


btn_run7.on_click(on_run7)
BADGES['7'] = widgets.HTML(badge_html('7'))
sec7_box = widgets.VBox([
    training_box(
        'Turns the flags of step 6 into a keep / reject decision, with tool 8bis\'s rule: drop the channels flagged '
        'in too many epochs of the reference stages, then reject every epoch flagged on too many of the remaining '
        'channels. Or shows a decision already taken by tool 8 (a reviewer, epoch by epoch) or tool 8bis.',
        'Tool 8bis decides automatically for every recording; tool 8 lets a reviewer decide. Both write the '
        'decision tables (derivatives/rejection_auto or rejection_manual/) that tool 9 reads with the epochs.',
        'Dropped channels and why; the share of rejected epochs per stage; would tool 8bis exclude this '
        'participant? Then the explorer: the decision strip shows each epoch kept or rejected, and the decision of '
        'the current epoch is written under the figures.'),
    BADGES['7'],
    widgets.HTML('<b>Decision source</b>'), dd_src7, html_src7, box_live7,
    btn_run7, out_sec7])
prefill_step7()


# ===== Step 8: built in the next phase of the tool-0 redesign =====
PLANNED = {
    '8': ('Spectral features on the kept epochs: power per band and stage, aperiodic (1/f) exponent and offset, '
          'oscillatory peaks.',
          'Tool 9 computes them for every recording and builds the database tables.',
          'Spectra per stage that look physiological (alpha in wake, spindles in N2, slow waves in N3).', 7),
}
planned_boxes = {}
for _k, (_what, _batch, _look, _phase) in PLANNED.items():
    BADGES[_k] = widgets.HTML(badge_html(_k))
    planned_boxes[_k] = widgets.VBox([
        training_box(_what, _batch, _look), BADGES[_k],
        widgets.HTML(f'<p style="color:#757575;"><i>This step is not built yet (tool-0 redesign, phase {_phase}). '
                     'Until then, run the batch tool named above.</i></p>')])

## Epoch explorer

In [ ]:
# ===== Epoch explorer: the night epoch by epoch, with tool 8's display =====
# Fixed clinical scales (µV per channel row) from the shared library: an amplitude keeps the same height from
# one epoch to the next, which is what makes visual criteria (75 µV slow waves, EMG tone...) usable.
_sclay, _scsty = widgets.Layout(width='120px'), {'description_width': '40px'}
ft_sc = {t: widgets.FloatText(description=t, value=L.DISPLAY_SCALE_UV[t], step=step, style=_scsty, layout=_sclay)
         for t, step in (('EEG', 25), ('EOG', 50), ('EMG', 25), ('ECG', 100))}
dd_signal = widgets.Dropdown(description='Signal:', options=[('as recorded', 'raw'),
                                                            ('display band-pass 0.3-35 Hz', 'bp')],
                             value='raw', style={'description_width': '55px'}, layout=widgets.Layout(width='300px'))
dd_nav_ch = widgets.Dropdown(description='Spectrogram:', options=[], style={'description_width': '90px'},
                             layout=widgets.Layout(width='230px'))
dd_channels = widgets.Dropdown(description='Channels:', value='all',
                               options=[('all', 'all'), ('flagged only', 'flagged'),
                                        ('flagged + neighbours', 'flagged+neighbours'), ('worst 16 (p-p)', 'worst')],
                               style={'description_width': '70px'}, layout=widgets.Layout(width='250px'))
sl_ctx = widgets.IntSlider(description='± context', value=1, min=0, max=3, layout=widgets.Layout(width='230px'))
cb_context = widgets.Checkbox(value=True, description='Show EOG/EMG/ECG', indent=False,
                              layout=widgets.Layout(width='170px'))
ft_epoch_offset = widgets.IntText(description='Epoch n° offset', value=1, style={'description_width': '110px'},
                                  layout=widgets.Layout(width='180px'))
sl_epoch = widgets.IntSlider(description='Epoch', value=0, min=0, max=0, continuous_update=False,
                             layout=widgets.Layout(width='520px'))
btn_prev = widgets.Button(description='◀ Prev', layout=widgets.Layout(width='90px'))
btn_next = widgets.Button(description='Next ▶', layout=widgets.Layout(width='90px'))
btn_next_stage = widgets.Button(description='Next stage change', layout=widgets.Layout(width='150px'))
lbl_epoch = widgets.HTML('')
SCORE_STAGES = ['W', 'N1', 'N2', 'N3', 'R', 'MT', '?']
stage_btns = {s: widgets.Button(description=s, button_style='warning', layout=widgets.Layout(width='52px'))
              for s in SCORE_STAGES}
cb_autoadv = widgets.Checkbox(value=True, description='Auto-advance after scoring', indent=False,
                              layout=widgets.Layout(width='210px'))
btn_save_hypno = widgets.Button(description='Save the rescored hypnogram', icon='save', button_style='success',
                                layout=widgets.Layout(width='250px'))
cb_overwrite = widgets.Checkbox(value=False, description='Overwrite an existing _rescored.txt', indent=False,
                                layout=widgets.Layout(width='260px'))
out_save = widgets.Output()
dd_annot_label = widgets.Dropdown(description='Label:', options=L.MANUAL_EVENT_LABELS,
                                  style={'description_width': '45px'}, layout=widgets.Layout(width='240px'))
ft_annot_offset = widgets.FloatText(description='at (s) in epoch', value=0.0, step=1.0,
                                    style={'description_width': '105px'}, layout=widgets.Layout(width='190px'))
txt_annot_comment = widgets.Text(description='Comment:', placeholder='free text: the event name when the label is "other"',
                                 style={'description_width': '70px'}, layout=widgets.Layout(width='380px'))
btn_annot_add = widgets.Button(description='Add annotation', icon='plus', layout=widgets.Layout(width='160px'))
box_annot_list = widgets.VBox([])
lbl_annot_info = widgets.HTML('')
out_nav = widgets.Output()
out_montage = widgets.Output()
out_detail = widgets.Output()
html_explorer_info = widgets.HTML('<i>Load a recording (step 1) to open the explorer.</i>')
KEY_TO_STAGE = {'w': 'W', '0': 'W', '1': 'N1', '2': 'N2', '3': 'N3', '4': 'R', '5': 'R', 'r': 'R', 'm': 'MT'}


def compute_nav_spectrogram():
    """Navigator spectrogram of the chosen channel: 30 s windows (50 % overlap) over the displayed signal,
    so its time axis is in epochs and a cursor move only redraws, never recomputes."""
    P = STATE['P']
    ch = dd_nav_ch.value if dd_nav_ch.value in P['ch_names'] else P['ch_names'][0]
    sig = P['data_uV'][:, P['ch_names'].index(ch), :].reshape(-1)
    nper = max(64, int(P['sfreq'] * 30))
    f, t, sxx = scipy_spectrogram(sig, fs=P['sfreq'], nperseg=nper, noverlap=nper // 2)
    keep = f <= 30
    sxx = sxx[keep]
    db = 10 * np.log10(np.where(sxx > 0, sxx, np.nan))       # µV²/Hz, log guarded (no epsilon floor)
    STATE['nav_spec'] = (t / 30.0, f[keep], db)


# The navigator and the montage are drawn at the same width with the same margins, then both stretched to
# the page width (up to 1500 px): the time series get bigger and stay aligned under the spectrogram.
FIG_LAYOUT = widgets.Layout(width='100%', max_width='1500px')
KEPT_COLOR, REJECTED_COLOR = '#2e7d32', '#c0392b'             # tool 8bis's epoch strip (its report)


def decision_on_grid():
    """Step 7's decision when it is on the explorer's 30 s epochs (step 6 run with 30 s epochs), else None."""
    d7, r6 = STATE.get('decision'), STATE.get('preproc')
    return d7 if d7 is not None and r6 is not None and r6['epoch_sec'] == 30 else None


def decision_html(ei):
    """The keep / reject decision of epoch `ei` (read-only: a decision is changed in tool 8)."""
    d7 = decision_on_grid()
    if d7 is None or ei >= len(d7['rejected']):
        return ''
    if d7['rejected'][ei]:
        why = d7['reasons'][ei]
        txt = f'<b style="color:{REJECTED_COLOR};">REJECTED</b>' + (f' ({escape(why)})' if why else '')
    else:
        txt = f'<b style="color:{KEPT_COLOR};">KEPT</b>'
    drop = (f"dropped channel(s): {escape(', '.join(d7['dropped']))}" if d7['dropped'] else 'no channel dropped')
    excl = (f' · <span style="color:#c62828;">tool 8bis would exclude the participant: {escape(d7["exclusion"])}'
            '</span>' if d7['exclusion'] else '')
    return (f"{txt} · {escape(d7['source_label'])} · {drop}{excl} <small>(read-only: a decision is changed in "
            'tool 8)</small>')


def plot_navigator(cur):
    """A thin events strip on top (scored events: ticks coloured by family; manual annotations: violet
    triangles), then the spectrogram of the night with the hypnogram (white line, REM in red), GSSC's hypnogram
    (dashed orange, optional), the rescored epochs (orange marks) and the current epoch (red cursor). The events
    have their own strip so they never hide the low frequencies. Once step 6 has run (30 s epochs), a second
    strip gives each epoch's flag, coloured by method (tool 7's reject_method: any channel); once step 7 has
    decided, a third one each epoch's decision (kept green, rejected red, tool 8bis's colours)."""
    n = STATE['n']
    labels = STATE['hypno_work'][:n]
    r6 = STATE.get('preproc')
    flags = r6 is not None and r6['epoch_sec'] == 30
    d7 = decision_on_grid()
    # Same width and left / right margins as tool 8's montage (plot_epoch_montage: 12 in, 14 on a dense
    # montage; plotting area from 17 % to 84 % of the width), so the two plots line up one above the other.
    width = 12.0 if len(STATE['P']['ch_names']) <= L.HD_CHANNEL_THRESHOLD else 14.0
    strips = ['events'] + (['flags'] if flags else []) + (['decision'] if d7 is not None else [])
    fig, axes = plt.subplots(len(strips) + 1, 1, figsize=(width, 2.65 + 0.25 * len(strips)), sharex=True,
                             gridspec_kw={'height_ratios': [1] * len(strips) + [9], 'hspace': 0.05})
    ax_ev, ax = axes[0], axes[-1]
    ax_fl = axes[strips.index('flags')] if flags else None
    ax_dec = axes[strips.index('decision')] if d7 is not None else None
    t_ep, f, db = STATE['nav_spec']
    im = None
    if np.isfinite(db).any():
        # tool 6's colour rule (quality_lib): the flat / disconnected columns left out of the scale and drawn
        # grey, the span capped at 45 dB, so a data gap never squeezes the sleep structure into a few colours
        img, norm, cmap = QL.spectrogram_colour_scale(db)
        im = ax.pcolormesh(t_ep, f, img, cmap=cmap, norm=norm, shading='auto', rasterized=True)
    ax.set_ylim(0.5, 30)
    ax.set_ylabel('Freq (Hz)', fontsize=8)
    ax.set_xlim(0, n)
    ax.set_xlabel('Epoch index', fontsize=8)
    ax.tick_params(labelsize=7)
    ax2 = ax.twinx()
    stage_y, _col, ytp, ytl = L.custom_stage_style(STATE['custom_stages'])
    y = [stage_y.get(s, min(ytp) - 1) for s in labels]
    outline = [mpe.Stroke(linewidth=3.4, foreground='black'), mpe.Normal()]
    ax2.step(np.arange(n + 1), y + [y[-1]], where='post', color='white', linewidth=1.7, zorder=6,
             path_effects=outline)
    # REM in red, at the line's own width (the usual PSG convention, kept discreet)
    for ei in np.where(np.asarray(labels) == 'R')[0]:
        ax2.plot([ei, ei + 1], [stage_y['R']] * 2, color=REM_COLOR, linewidth=1.7, solid_capstyle='butt',
                 zorder=6.5, path_effects=outline)
    ax2.set_yticks(ytp)
    ax2.set_yticklabels(ytl, fontsize=8)
    ax2.set_ylim(min(ytp) - 1.0, max(ytp) + 1.0)
    ax2.set_xlim(0, n)
    legend = [Line2D([0], [0], color='white', lw=1.7, path_effects=outline, label='hypnogram in use')]
    rescored = np.where(np.asarray(labels) != np.asarray(STATE['hypno_orig'][:n]))[0]
    for ei in rescored:
        ax2.plot([ei + 0.5], [max(ytp) + 0.7], marker='v', color='#e67e22', markersize=7, zorder=8)
    if len(rescored):
        legend.append(Line2D([0], [0], marker='v', color='#e67e22', linestyle='none', label='rescored'))
    gs = STATE.get('gssc')
    if gs is not None and cb_gssc_nav.value:   # GSSC's hypnogram (step 3), dashed orange under the white line
        m = min(n, len(gs['stages']))
        yg = [stage_y.get(s, np.nan) for s in gs['stages'][:m]]
        ax2.step(np.arange(m + 1), yg + [yg[-1]], where='post', color='#ff9800', linewidth=1.2,
                 linestyle='--', zorder=7)
        legend.append(Line2D([0], [0], color='#ff9800', lw=1.2, linestyle='--', label='GSSC (step 3)'))
    # events strip: one tick per scored event (colour = family), violet triangles = manual annotations
    ax_ev.set_ylim(0, 1)
    ax_ev.set_yticks([])
    ax_ev.set_ylabel('events', fontsize=7, rotation=0, ha='right', va='center')
    for sp in ('top', 'right', 'left'):
        ax_ev.spines[sp].set_visible(False)
    ax_ev.tick_params(axis='x', bottom=False)
    families = {}
    on = STATE.get('onsets')
    if on is not None and len(on):
        for typ, grp in on.groupby('type'):
            fam, col = event_family(typ)
            families[fam] = col
            ax_ev.plot(grp['onset_s'].values / 30.0, np.full(len(grp), 0.5), linestyle='none', marker='|',
                       color=col, markersize=7, markeredgewidth=1.2)
    man = STATE.get('manual_events')
    if man is not None and len(man):
        ax_ev.plot(pd.to_numeric(man['onset_s'], errors='coerce').values / 30.0, np.full(len(man), 0.5),
                   linestyle='none', marker='v', color=L.MANUAL_COLOR, markersize=5)
    for fam, col in families.items():
        legend.append(Line2D([0], [0], marker='|', color=col, markeredgewidth=1.5, linestyle='none', label=fam))
    if man is not None and len(man):
        legend.append(Line2D([0], [0], marker='v', color=L.MANUAL_COLOR, linestyle='none', label='annotation'))
    ax_ev.axvline(cur + 0.5, color='#ff1744', linewidth=2.5)
    ax2.axvline(cur + 0.5, color='#ff1744', linewidth=2.5, zorder=10)
    if flags:
        # step 6's flag of each epoch, coloured by method (the toolkit's palette, qc_rejected_epochs_lib)
        methods = r6['epochs'].metadata['reject_method'].astype(str).values[:n]
        codes = np.array([PL.MULTIPLE_CODE if m == 'multiple' else PL.METHOD_CODE.get(m, 0) for m in methods])
        ax_fl.imshow(np.ma.masked_equal(codes[np.newaxis, :], 0), aspect='auto', interpolation='nearest',
                     cmap=mcolors.ListedColormap(L.HEATMAP_COLORS), vmin=-0.5, vmax=len(L.HEATMAP_COLORS) - 0.5,
                     extent=(0, len(codes), 0, 1))
        ax_fl.set_ylim(0, 1)
        ax_fl.set_yticks([])
        ax_fl.set_ylabel('flags', fontsize=7, rotation=0, ha='right', va='center')
        for sp in ('top', 'right', 'left'):
            ax_fl.spines[sp].set_visible(False)
        ax_fl.tick_params(axis='x', bottom=False)
        ax_fl.axvline(cur + 0.5, color='#ff1744', linewidth=2.5)
        for m in [m for m in PL.METHOD_ORDER if m in set(methods)] + (['multiple'] if 'multiple' in set(methods) else []):
            code = PL.MULTIPLE_CODE if m == 'multiple' else PL.METHOD_CODE[m]
            legend.append(Line2D([0], [0], marker='s', color=L.HEATMAP_COLORS[code], linestyle='none',
                                 label=PL.METHOD_LABEL.get(m, m)))
    if d7 is not None:
        # step 7's decision of each epoch, in tool 8bis's colours (its report's epoch strip)
        rej = np.asarray(d7['rejected'][:n], dtype=float)
        ax_dec.imshow(rej[np.newaxis, :], aspect='auto', interpolation='nearest', vmin=0, vmax=1,
                      cmap=mcolors.ListedColormap([KEPT_COLOR, REJECTED_COLOR]), extent=(0, len(rej), 0, 1))
        ax_dec.set_ylim(0, 1)
        ax_dec.set_yticks([])
        ax_dec.set_ylabel('decision', fontsize=7, rotation=0, ha='right', va='center')
        for sp in ('top', 'right', 'left'):
            ax_dec.spines[sp].set_visible(False)
        ax_dec.tick_params(axis='x', bottom=False)
        ax_dec.axvline(cur + 0.5, color='#ff1744', linewidth=2.5)
        legend.append(Line2D([0], [0], marker='s', color=KEPT_COLOR, linestyle='none', label='kept'))
        legend.append(Line2D([0], [0], marker='s', color=REJECTED_COLOR, linestyle='none', label='rejected'))
    # the legend may take two rows: the top margin is sized for it (in inches, whatever the figure height)
    height = fig.get_figheight()
    ncol = min(len(legend), 8)
    n_rows = int(np.ceil(len(legend) / ncol))
    fig.subplots_adjust(left=0.17, right=0.84, top=1 - (0.12 + 0.17 * n_rows) / height, bottom=0.49 / height)
    # the colour scale in the right margin, beside the stage labels (the plotting area keeps the montage's
    # width and margins, so the two stay aligned)
    if im is not None:
        pos = ax.get_position()
        cbar = fig.colorbar(im, cax=fig.add_axes([0.878, pos.y0, 0.007, pos.height]))
        cbar.set_label('dB (µV²/Hz)', fontsize=7)
        cbar.ax.tick_params(labelsize=6)
    # the legend above the strips, right-aligned on the plotting area
    fig.legend(handles=legend, loc='lower right', bbox_to_anchor=(0.84, 1 - (0.08 + 0.17 * n_rows) / height),
               ncol=ncol, fontsize=6.5, frameon=False, handlelength=1.6, columnspacing=1.2, borderaxespad=0)
    return fig


def preview_thresholds():
    """The thresholds the montage colours the traces with: step 6's run when its preprocessed signal is shown;
    otherwise the step-6 tab's settings (tool 7's defaults unless changed), a preview on the displayed signal."""
    if STATE.get('display_mode') == 'preproc' and STATE.get('preproc') is not None:
        return STATE['preproc']['thresholds']
    return step6_thresholds(preproc_settings())


def explorer_info_html():
    """The note above the explorer: what the trace colours mean, and what step 6 adds."""
    r6 = STATE.get('preproc')
    if STATE.get('display_mode') == 'preproc':
        colours = ("Signal and flags of step 6 (tool 7's preprocessing), shown as tool 8 shows them: the trace "
                   "colours are recomputed with step 6's thresholds, the strip above the spectrogram gives each "
                   'epoch\'s flag.')
    elif r6 is not None and r6['epoch_sec'] == 30:
        colours = ("Trace colours preview tool 7's criteria at the step-6 tab's thresholds on the displayed signal; "
                   'choose <b>Signal: preprocessed + flags (step 6)</b> to see step 6\'s signal and flags.')
    elif r6 is not None:
        colours = (f"Step 6 ran with {r6['epoch_sec']} s epochs: the explorer keeps its 30 s grid, so its "
                   "preprocessed view needs 30 s epochs. Trace colours preview tool 7's criteria on the displayed "
                   'signal.')
    else:
        colours = ("Trace colours preview tool 7's criteria at the step-6 tab's thresholds (tool 7's defaults) on "
                   'the displayed signal: step 6 computes the real flags.')
    if decision_on_grid() is not None:
        colours += (' The decision strip (step 7) shows each epoch kept (green) or rejected (red); the decision of '
                    'the current epoch is written under the figures.')
    return ('<small>Fixed clinical scales (µV per row), so amplitudes compare from one epoch to the next. '
            + colours + ' PSD background: median and IQR of the epochs of the same stage. '
            + ('Keyboard (click the navigator or the montage first): ←/→ epochs, w/0 = W, 1, 2, 3, r/4/5 = R, m = MT.'
               if _HAS_IPYEVENTS else 'Install ipyevents for keyboard shortcuts.') + '</small>')


SIGNAL_OPTIONS = [('as recorded', 'raw'), ('display band-pass 0.3-35 Hz', 'bp')]


def set_signal_options(with_preproc):
    """The explorer's signal choices: step 6's preprocessed signal is offered once step 6 ran (30 s epochs)."""
    dd_signal.unobserve(_on_signal, names='value')
    dd_signal.options = SIGNAL_OPTIONS + ([('preprocessed + flags (step 6)', 'preproc')] if with_preproc else [])
    mode = STATE.get('display_mode', 'raw')
    dd_signal.value = mode if (mode != 'preproc' or with_preproc) else 'raw'
    dd_signal.observe(_on_signal, names='value')


def explorer_after_step6():
    """Step 6 ran: offer its preprocessed signal (30 s epochs only: the explorer's grid), show the new flags
    when that signal is on screen, and redraw (the navigator gains the flag strip)."""
    r6 = STATE['preproc']
    ok = r6['epoch_sec'] == 30
    if STATE.get('display_mode') == 'preproc':
        build_display('preproc' if ok else 'raw')
    set_signal_options(ok)
    html_explorer_info.value = explorer_info_html()
    render_explorer()


def explorer_after_step7():
    """Step 7 decided: the navigator gains the decision strip, the box under the figures the epoch's decision."""
    html_explorer_info.value = explorer_info_html()
    render_explorer()


def epoch_identity(ei):
    num, elapsed, clock = L.epoch_labels(ei, 30, STATE['P'].get('meas_date'), int(ft_epoch_offset.value))
    return num, elapsed, clock


def clock_at(onset_s):
    md = STATE['P'].get('meas_date')
    if md is None:
        return ''
    try:
        return (md + pd.to_timedelta(float(onset_s), unit='s')).strftime('%H:%M:%S')
    except Exception:
        return ''


def status_html(ei):
    num, elapsed, clock = epoch_identity(ei)
    stage, orig = STATE['hypno_work'][ei], STATE['hypno_orig'][ei]
    changed = f' <span style="color:#e65100;">(rescored, was {orig})</span>' if stage != orig else ''
    n_mod = int((STATE['hypno_work'] != STATE['hypno_orig']).sum())
    clock_txt = f' · clock {clock}' if clock else ''
    n_unsaved = unsaved_rescoring()
    unsaved = (f' <span style="color:#e65100;">({n_unsaved} not saved)</span>' if n_unsaved else '')
    gssc_txt = ''
    gs = STATE.get('gssc')
    if gs is not None and ei < len(gs['stages']):
        g = gs['stages'][ei]
        col = '#1b5e20' if g == stage else '#c62828'
        gssc_txt = (f' · GSSC <b style="color:{col};">{g}</b> ({gs["probs"][ei].max() * 100:.0f} % sure)')
    return (f'<b>Epoch {ei} (#{num})</b> · {elapsed} since the start{clock_txt} · stage <b>{stage}</b>{changed}'
            f'{gssc_txt} · {n_mod} epoch(s) rescored{unsaved}')


def next_disagreement(b=None):
    """Next epoch where GSSC and the hypnogram in use disagree (W/N1/N2/N3/R epochs only)."""
    gs = STATE.get('gssc')
    if 'P' not in STATE or gs is None:
        return
    m = min(STATE['n'], len(gs['stages']))
    work = STATE['hypno_work']
    hit = next((j for j in range(STATE['cur'] + 1, m)
                if work[j] in L.AASM_STAGES and gs['stages'][j] != work[j]), None)
    if hit is None:
        lbl_epoch.value = status_html(STATE['cur']) + ' · <i>no disagreement after this epoch</i>'
    else:
        goto_epoch(hit)


def use_gssc_stage(b=None):
    gs = STATE.get('gssc')
    if 'P' in STATE and gs is not None and STATE['cur'] < len(gs['stages']):
        score_current(str(gs['stages'][STATE['cur']]))


def render_explorer():
    if 'P' not in STATE:
        return
    ei = STATE['cur']
    thr = preview_thresholds()
    P = STATE['P']
    lbl_epoch.value = status_html(ei)
    html_decision.value = decision_html(ei)
    box_decision.layout.display = '' if html_decision.value else 'none'
    with out_nav:
        clear_output(wait=True)
        try:
            display(widgets.Image(value=fig_to_png(plot_navigator(ei), tight=False), format='png',
                                  layout=FIG_LAYOUT))
        except Exception as e:
            print(f'Navigator error: {e}')
    with out_montage:
        clear_output(wait=True)
        try:
            num, _elapsed, clock = epoch_identity(ei)
            scored, manual = L.merge_onsets_for_display(STATE.get('onsets'), STATE.get('manual_events'))
            show = None if dd_channels.value == 'all' else L.select_montage_channels(P, ei, thr, mode=dd_channels.value)
            fig = L.plot_epoch_montage(P, ei, thr, context=int(sl_ctx.value),
                                       ctx=STATE['ctx'] if cb_context.value else None,
                                       onsets=scored,
                                       title_method=(None if STATE.get('display_mode') == 'preproc' else
                                                     'preview' if STATE.get('preproc') is not None else
                                                     'step 6 not run'),
                                       scales={t: w.value for t, w in ft_sc.items()}, show_channels=show,
                                       number_label=f'(#{num}' + (f', {clock}' if clock else '') + ')',
                                       manual=manual, clock_start=P.get('meas_date'))
            # drawn without a tight bounding box (alignment), so leave room for the time axis and its clock row
            fig.subplots_adjust(bottom=0.8 / fig.get_figheight())
            display(widgets.Image(value=fig_to_png(fig, tight=False), format='png', layout=FIG_LAYOUT))
        except Exception as e:
            print(f'Montage error: {e}')
    with out_detail:
        clear_output(wait=True)
        try:
            fig = L.plot_epoch_detail(P, ei, STATE['freqs'], STATE['psds'], thr, fmin=2.0)
            display(widgets.Image(value=fig_to_png(fig), format='png',
                                  layout=widgets.Layout(max_width='100%')))
        except Exception as e:
            print(f'Detail error: {e}')
    refresh_annot_list(ei)


def goto_epoch(i):
    if 'P' not in STATE:
        return
    i = int(np.clip(i, 0, STATE['n'] - 1))
    STATE['cur'] = i
    if sl_epoch.value != i:
        sl_epoch.value = i                    # its observer redraws
    else:
        render_explorer()


def _on_slider(change):
    if 'P' in STATE:
        STATE['cur'] = int(change['new'])
        render_explorer()


def _next_stage_change(b):
    if 'P' not in STATE:
        return
    labels, cur = STATE['hypno_work'], STATE['cur']
    nxt = next((j for j in range(cur + 1, STATE['n']) if labels[j] != labels[cur]), STATE['n'] - 1)
    goto_epoch(nxt)


def score_current(stage):
    """Assign a stage to the current epoch (stage buttons and keyboard). Held in memory until saved; the
    steps computed from the hypnogram become out of date."""
    if 'P' not in STATE:
        return
    ei = STATE['cur']
    if STATE['hypno_work'][ei] != stage:
        STATE['hypno_work'][ei] = stage
        STATE['P']['stages'][ei] = stage
        STATE['P']['meta'].at[ei, 'stage'] = stage
        mark_out_of_date(AFTER_RESCORING, 'the hypnogram was rescored')
    if cb_autoadv.value and ei < STATE['n'] - 1:
        goto_epoch(ei + 1)
    else:
        render_explorer()


def on_save_hypno(b):
    with out_save:
        clear_output()
        try:
            if 'P' not in STATE:
                print('Load a recording first.')
                return
            base = STATE['hypno_path'] if STATE['hypno_path'] is not None else Path(STATE['edf'])
            out_txt = base.with_name(f'{base.stem}_rescored.txt')
            out_log = base.with_name(f'{base.stem}_rescore_log.tsv')
            if out_txt.exists() and not cb_overwrite.value:
                print(f'⚠ {out_txt.name} already exists: tick "Overwrite an existing _rescored.txt" to replace it.')
                return
            work, orig = STATE['hypno_work'], STATE['hypno_orig']
            np.savetxt(str(out_txt), work, fmt='%s')
            STATE['hypno_saved'] = work.copy()
            STATE['discard_armed'] = None
            html_guard.value = ''
            now = datetime.now().isoformat(timespec='seconds')
            rows = [{'epoch_idx': int(ei), 'epoch_time_sec': int(ei * 30), 'original_label': str(orig[ei]),
                     'new_label': str(work[ei]), 'timestamp': now} for ei in np.where(work != orig)[0]]
            pd.DataFrame(rows, columns=['epoch_idx', 'epoch_time_sec', 'original_label', 'new_label',
                                        'timestamp']).to_csv(str(out_log), sep='\t', index=False)
            print(f'Saved {out_txt.name} ({len(work)} epochs, {len(rows)} rescored) and {out_log.name} '
                  f'in {out_txt.parent}. The original hypnogram is untouched.')
            REPORT_BLOCKS.setdefault('explorer', [])
            REPORT_BLOCKS['explorer'] = [b_ for b_ in REPORT_BLOCKS['explorer'] if 'id="rescoring"' not in b_]
            REPORT_BLOCKS['explorer'].append(f'<p id="rescoring">Rescoring saved to <code>{out_txt}</code>: '
                                             f'{len(rows)} epoch(s) changed.</p>')
            log_action('explorer', 'rescored hypnogram saved', f'{out_txt.name}: {len(rows)} epoch(s) changed')
        except Exception as e:
            print(f'Save failed: {e!r}')


# ----- Manual annotations: the same file and format as tool 8, beside the recording -----
def annot_label_options():
    opts = [v for v in (STUDY.get('event_remap') or {}).values() if isinstance(v, str) and v]
    opts += L.MANUAL_EVENT_LABELS
    return list(dict.fromkeys(opts))


def annots_of_epoch(ei):
    df = STATE.get('manual_events')
    if df is None or not len(df):
        return []
    return list(df.index[pd.to_numeric(df['epoch_idx'], errors='coerce') == int(ei)])


def save_annots():
    p = L.save_manual_events(Path(STATE['edf']).parent, STATE['fid'], STATE['manual_events'])
    if p is None:
        lbl_annot_info.value = '<span style="color:#c62828;">⚠ could not write the annotation file.</span>'
    else:
        lbl_annot_info.value = f"<small>{len(STATE['manual_events'])} annotation(s) &rarr; {p}</small>"
        mark_out_of_date(AFTER_ANNOTATION, 'an annotation was added or deleted')
        man = STATE['manual_events']
        REPORT_BLOCKS.setdefault('explorer', [])
        REPORT_BLOCKS['explorer'] = [b_ for b_ in REPORT_BLOCKS['explorer'] if 'id="annotations"' not in b_]
        REPORT_BLOCKS['explorer'].append(f'<div id="annotations"><p>Manual annotations ({len(man)}), in '
                                         f'<code>{p}</code>:</p>'
                                         + df_html(man[['type', 'onset_s', 'epoch_number', 'clock_time', 'stage',
                                                        'comment', 'source']]) + '</div>')


def _make_annot_delete(row_id):
    def _f(b):
        try:
            row = STATE['manual_events'].loc[row_id]
            STATE['manual_events'] = STATE['manual_events'].drop(index=row_id)
            save_annots()
            log_action('explorer', 'annotation deleted', f"{row['type']} at {float(row['onset_s']):.1f} s")
            render_explorer()
        except Exception as e:
            lbl_annot_info.value = f'<span style="color:#c62828;">Delete error: {e}</span>'
    return _f


def refresh_annot_list(ei):
    try:
        rows = []
        for rid in annots_of_epoch(ei):
            a = STATE['manual_events'].loc[rid]
            cmt = str(a.get('comment', '') or '')
            cmt = '' if cmt.lower() == 'nan' else cmt
            txt = (f'<b>{a["type"]}</b> @ {float(a["onset_s"]):.1f} s {a.get("clock_time", "")} '
                   f'<small>({a.get("source", "")})</small>' + (f': {cmt}' if cmt else ''))
            btn = widgets.Button(description='', icon='trash', tooltip='Delete this annotation',
                                 layout=widgets.Layout(width='40px'))
            btn.on_click(_make_annot_delete(rid))
            rows.append(widgets.HBox([btn, widgets.HTML(txt)]))
        box_annot_list.children = tuple(rows)
    except Exception as e:
        box_annot_list.children = (widgets.HTML(f'<span style="color:#c62828;">Annotation list error: {e}</span>'),)


def on_annot_add(b):
    # Written to disk at once: a Voila session that dies must never take an annotation with it.
    try:
        if 'P' not in STATE:
            lbl_annot_info.value = '<span style="color:#c62828;">Load a recording first.</span>'
            return
        ei = STATE['cur']
        label = str(dd_annot_label.value)
        comment = txt_annot_comment.value.strip()
        if label == 'other':
            if not comment:
                lbl_annot_info.value = ('<span style="color:#e65100;">Type the event name in Comment when the '
                                        'label is "other".</span>')
                return
            label = comment
        off = min(max(float(ft_annot_offset.value), 0.0), 30.0)    # the mark stays inside its own epoch
        onset = ei * 30.0 + off
        num, _elapsed, _clock = epoch_identity(ei)
        row = {'type': label, 'onset_s': round(onset, 3), 'duration_s': np.nan, 'epoch_idx': int(ei),
               'epoch_number': num, 'clock_time': clock_at(onset), 'stage': STATE['hypno_work'][ei],
               'comment': comment, 'source': 'manual_tool0',
               'created_at': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
        new = pd.DataFrame([row])
        STATE['manual_events'] = (new if not len(STATE['manual_events'])
                                  else pd.concat([STATE['manual_events'], new], ignore_index=True))
        save_annots()
        txt_annot_comment.value = ''
        log_action('explorer', 'annotation added', f'{label} at {onset:.1f} s (epoch {ei})')
        render_explorer()
    except Exception as e:
        lbl_annot_info.value = f'<span style="color:#c62828;">Annotation error: {e}</span>'


def explorer_open():
    """Called once step 1 has loaded a recording: sync the widgets and draw the first epoch."""
    P = STATE['P']
    dd_nav_ch.unobserve(_on_nav_ch, names='value')
    dd_nav_ch.options = P['ch_names']
    dd_nav_ch.value = P['ch_names'][0]
    dd_nav_ch.observe(_on_nav_ch, names='value')
    set_signal_options(False)                  # a (re)loaded recording: step 6 has to run again
    dd_annot_label.options = annot_label_options()
    sl_epoch.unobserve(_on_slider, names='value')
    sl_epoch.max = max(0, STATE['n'] - 1)
    sl_epoch.value = 0
    sl_epoch.observe(_on_slider, names='value')
    STATE['cur'] = 0
    html_explorer_info.value = explorer_info_html()
    lbl_annot_info.value = (f"<small>{len(STATE['manual_events'])} annotation(s) in "
                            f"<code>{STATE['fid']}_manual_events.tsv</code> beside the recording.</small>")
    render_explorer()


def explorer_reset():
    for o in (out_nav, out_montage, out_detail, out_save):
        o.clear_output()
    lbl_epoch.value = ''
    box_annot_list.children = ()
    lbl_annot_info.value = ''
    box_gssc_nav.layout.display = 'none'
    html_explorer_info.value = '<i>Load a recording (step 1) to open the explorer.</i>'


def _on_nav_ch(change):
    if 'P' in STATE:
        compute_nav_spectrogram()
        render_explorer()


def _on_signal(change):
    if 'P' not in STATE:
        return
    try:
        lbl_epoch.value = '<i>Computing the displayed signal…</i>'
        cur = STATE['cur']
        build_display(change['new'])
        STATE['cur'] = min(cur, STATE['n'] - 1)
        render_explorer()
    except Exception as e:
        lbl_epoch.value = f'<span style="color:#c62828;">Display error: {e}</span>'


btn_prev.on_click(lambda b: goto_epoch(STATE.get('cur', 0) - 1))
btn_next.on_click(lambda b: goto_epoch(STATE.get('cur', 0) + 1))
btn_next_stage.on_click(_next_stage_change)
btn_next_disagree = widgets.Button(description='Next disagreement', icon='forward',
                                   tooltip='Next epoch where GSSC and the hypnogram in use disagree',
                                   layout=widgets.Layout(width='170px'))
btn_use_gssc = widgets.Button(description="Use GSSC's stage", layout=widgets.Layout(width='150px'),
                              tooltip='Rescore the current epoch with the stage GSSC gives it')
btn_next_disagree.on_click(next_disagreement)
btn_use_gssc.on_click(use_gssc_stage)
cb_gssc_nav = widgets.Checkbox(value=True, description='Show it on the navigator', indent=False,
                               layout=widgets.Layout(width='190px'))
box_gssc_nav = widgets.HBox([widgets.HTML('<b>GSSC (step 3, dashed orange):</b>'), btn_next_disagree, btn_use_gssc,
                             cb_gssc_nav], layout=widgets.Layout(display='none'))
for _s, _b in stage_btns.items():
    _b.on_click(lambda b, st=_s: score_current(st))
btn_save_hypno.on_click(on_save_hypno)
btn_annot_add.on_click(on_annot_add)
sl_epoch.observe(_on_slider, names='value')
dd_nav_ch.observe(_on_nav_ch, names='value')
dd_signal.observe(_on_signal, names='value')
for _w in list(ft_sc.values()) + [dd_channels, sl_ctx, cb_context, ft_epoch_offset, cb_gssc_nav]:
    _w.observe(lambda c: render_explorer(), names='value')

# Keep / reject decision of the current epoch, RIGHT UNDER the figures as in tool 8. Tool 0 only shows it
# (changing a decision is tool 8's job): filled once step 7 has decided, hidden until then.
html_decision = widgets.HTML('')
box_decision = widgets.HBox([widgets.HTML('<b>Decision:</b>'), html_decision],
                            layout=widgets.Layout(display='none'))
_box_style = dict(border='1px solid #ddd', padding='6px', margin='6px 0 6px 0', overflow='hidden')
box_rescore = widgets.VBox([
    widgets.HTML('<b>Rescore this epoch</b> <small>(held in memory until saved; the original hypnogram is '
                 'never overwritten)</small>'),
    widgets.HBox(list(stage_btns.values()) + [cb_autoadv]),
    box_gssc_nav,
    widgets.HBox([btn_save_hypno, cb_overwrite]), out_save], layout=widgets.Layout(**_box_style))
box_annot = widgets.VBox([
    widgets.HTML('<b>Annotate this epoch</b> <small>(e.g. an arousal the scorer missed): saved beside the '
                 'recording in <code>{file_id}_manual_events.tsv</code>, the same file as tool 8</small>'),
    widgets.HBox([dd_annot_label, ft_annot_offset, txt_annot_comment, btn_annot_add]),
    box_annot_list, lbl_annot_info], layout=widgets.Layout(**_box_style))

# Order = how a night is reviewed (as tool 8): navigation RIGHT ABOVE the figures, the decision RIGHT UNDER
# them, then rescoring and annotations, and the display settings last (rarely touched during a review).
figures_box = widgets.VBox([out_nav, out_montage, out_detail])   # the keyboard listens here (no text field)
explorer_box = widgets.VBox([
    html_explorer_info,
    widgets.HBox([btn_prev, sl_epoch, btn_next, btn_next_stage, ft_epoch_offset]),
    lbl_epoch,
    figures_box,
    box_decision,
    box_rescore,
    box_annot,
    widgets.HTML('<b>Display settings</b>'),
    widgets.HBox([dd_signal, dd_nav_ch, dd_channels]),
    widgets.HBox([sl_ctx, cb_context]),
    widgets.HBox([widgets.HTML('<b>Scale (µV/row):</b>')] + list(ft_sc.values()))])

if _HAS_IPYEVENTS:
    _kb = _IpyEvent(source=figures_box, watched_events=['keydown'], prevent_default_action=True)

    def _on_key(event):
        k = event.get('key', '')
        if k == 'ArrowRight':
            goto_epoch(STATE.get('cur', 0) + 1)
        elif k == 'ArrowLeft':
            goto_epoch(STATE.get('cur', 0) - 1)
        elif k.lower() in KEY_TO_STAGE:
            score_current(KEY_TO_STAGE[k.lower()])

    _kb.on_dom_event(_on_key)

In [ ]:
# ===== Page layout =====
def reset_participant():
    """A new participant (or study): every step and the explorer start from scratch."""
    reset_state()
    for o in (out_sec1, out_sec2, out_sec3, out_gssc, out_sec4, out_sec5, out_ch5, out_sec6, out_sec7):
        o.clear_output()
    prefill_step7()
    acc_ch5.layout.display = 'none'
    html_prog5.value = ''
    html_prog6.value = ''
    html_ch6.value = ''
    CH6.clear()
    box_ch6.children = ()
    set_signal_options(False)
    html_gssc.value = ''
    html_part.value = ''
    html_cfg1.value = ''
    html_ctx1.value = ''
    html_hypno1.value = ''
    html_report_info.value = ''
    html_guard.value = ''
    box_eeg1.children = ()
    explorer_reset()


TABS = widgets.Tab(children=[sec1_box, sec2_box, sec3_box, sec4_box, sec5_box, sec6_box, sec7_box,
                             planned_boxes['8']])
refresh_badges()
explorer_acc = widgets.Accordion(children=[explorer_box])
explorer_acc.set_title(0, 'Epoch explorer')
explorer_acc.selected_index = 0
display(fc_data, html_study, widgets.HBox([dd_part, cb_report]), html_part, html_guard, html_report_info,
        HTML('<h3>Steps</h3>'), TABS, HTML('<h3>Epoch explorer</h3>'), explorer_acc)